# Deney 06 — Render Pipeline (Notebook 01+02+03 çıktılarını birleştirir)

**Amaç:** Onaylanmış an + script + seslendirmeyi kullanarak dikey (9:16) bir YouTube Shorts videosu üretmek: kaynak aralığı kes, dikey kadraj uygula, seslendirmeyi orijinal sesle miksle (ducking), altyazı yak, kaliteli kodlama ile dışa aktar.

**Çalıştırma sırası:** Notebook 01 → an onayı → Notebook 02 → script onayı → Notebook 03 → seslendirme onayı → **bu notebook**. Alternatif olarak `00_youtube_automation_master.ipynb` tek başına tüm fazları çalıştırabilir.

**Bu notebook hiçbir yere yükleme/yayınlama yapmaz.** Çıktı sadece `results/render/final_short.mp4` olarak yerelde/Colab oturumunda kalır.

**Tek kaynak politikası (düzeltildi):** Bu notebook artık ffmpeg filtergraph'larını, altyazı üretimini, doğrulamayı ve onay kapısı mantığını **kendi içinde tekrar tanımlamaz**. Hepsi `local_pipeline/render_utils.py` içindeki kanonik fonksiyonlardan gelir — master notebook (`00_...`) ve `verify_local.py` de **aynı** fonksiyonları çağırır, yani üç ayrı kopya yoktur. Modül aşağıdaki bootstrap hücresiyle yüklenir (repo → Drive → `PROJECT_REPO_URL` → gömülü hash-doğrulamalı yedek).

**Onay politikası (üretimden sonra, içeriğe bağlı):** `run_render_pipeline()` **onay argümanı almaz** ve asla `approved` üretemez — yeni render her zaman `review_required` (teknik doğrulama başarısızsa `failed`) olur. Onay, videoyu **izledikten sonra** bölüm 5'teki ayrı hücrede, dosyanın SHA-256'sı doğrulanarak verilir. Yeniden render alırsan hash değişir ve önceki onay otomatik geçersizleşir. `render_validation` bir **teknik** kontrol sonucudur, insan onayı değildir.

In [ ]:
!pip -q install imageio-ffmpeg faster-whisper pandas > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, json
from pathlib import Path

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RENDER_DIR = RESULTS_DIR / 'render'
RENDER_DIR.mkdir(parents=True, exist_ok=True)

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}
print('Sonuc dizini:', RESULTS_DIR)


## `local_pipeline` kanonik modüllerini yükle (tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['render_utils', 'video_analysis', 'approval']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'render_utils': {
        'sha256': 'e6c7ff3f7c47b499702730b8407fe7947c13fd460942d044e326740418de8a7e',
        'encoding': 'zlib+b85',
        'data': (
            'c-rlK+m_olmgswZ1yWfvloFQe;!E6eD_M>&9iMjWESHl^uW}uklBg0}6shK=%5sg?nm;%%^Eu~h<_FF%oP7a600dQK$2~pM)9#K<'
            'k-)}nV`Ja3zrX)FuFGhW#^Twt)jGZqMV!sz;$WSu<21=)v53=kT$G|*L`6IklZ`mKKjuPXRcMf`H`zq=t`^C3v6p1{t`t!w;=k0%'
            'Wt7HQ6^h4s8chxosJO=3StX|VdL#0=s1`{n()=Qs_My^DT*gH*-;C3I8l{8vW{=^DWVOzVs^sFUd{#rJXGsxHt8^pgc>y0{L6zhn'
            'M7heBv8dv*>JRqzUgl6UyMPfDv8dN+9?ePt)z<YSP09uJctb@7;&~<V48RsK7Gj|DB&9MlaaAM$bkBTUQ-hI!Hz`bCS+Cb=0`ryE'
            ')w-_4I;s|fz5V_Dy?K$Z#CSZftGbBCV>v1So8?thC3#lv?a6nGs9dDUME(48nP=*`h}BcMsH-H^zv{`l$ft2xsyEdF#{h%kmS<5F'
            'SIH_??W*6Qz`sA_S<L0I!_c-W_7nWv+dF;p=*`ck<6j=V`1$Fn7>VmW0e|-6_kbZ#JFB8}KNS13_%crOHNF_H(`Xtm@)TwVKNaz1'
            '5?=xS0Ac_-?S~8@T0^fd@weDdva+7flWBt0p?eX&oJTOr^cf1y>M2gt7zefi5L@Wk&)>g!{^rHg@vlETKmF<H>+z4To;-a)eGIB3'
            '+XSHq$`z~)KCI$dQm^nSjf#u-;PTD^EXE`c0MO${KfQVW>gD*&4^Mx5I{xYPt7p$&7-J1)@jR;2Dj3Qa^b-1nSTF{}EumxZ@i=V0'
            '&T|+@7Oi4<A3TZw9Q{&@(<mzgw@jIQz%t*Q7#|`S>Sa8h=4oDxqq0O8|NO%d{CNg{Oc0Z(^ptz{%q_Qy-jA>HVunTTI`5~j$?^T2'
            'BkMzPk!0gbcz<$YegGVgizze<U!Fy285`fyxT=6cl<Q~;xCaGf#V!96T)=#c^Z6Li6L7A)X+BT7=JUs>n8s;Zm*Vts=Y$^p%_sEa'
            '!N*SMo1)%5p92$QRRBUwi)2k31LkkFMjY+w5>)C*6tD73yn*(kViRYfc)f_?Y*R$$yI+zrE;gn4>c>qP6-fj;#s>I9UYD_1@K>O<'
            'agmpycv3_cK+Gyw-BA1y`usL1#S5UG@YsU$yufY|DnR>Dx`D>9Coyc$4IqFn-FKH5@uY<dr;iU_#}_qJ9#qx5jTVdO<lcR(adhYC'
            '?%il||K7>TgUQjodv~Y5i|$R2<NJ4J^ON7r?#A~XMBf}gxc5zba(wdO=s1ez-`syNo6nAq?oVd(qZa78ND=O$s@CQ3@UV!k1{X=S'
            's3$N_#S}^aQ5#J2)!{{+U!?INRw@ryQIZ|z^AsNB2+HZ<Ek=_j(>MbpH7DWKvloNvedR0m89V1`WdcVa&}g8_X+&ELj}UARHct$2'
            '6f_Cs;C4e)P}LYd0=nHtNqD{#e~i~~+%kLe9uDbgf}F@|4G0bg>|v3o=_HyiMNWHZARg6KzJfNV;t#Lhhyo6DWFoTsO2UK5D~a`3'
            '<-k|K0bC_j8Xw@{M0}M=p@gJtBAUQ)H}I0eF>LS;ar((8%ww+JK@0Cc_}kOMZ$39IJRY3vq6<7D2frW*gB7`fCd}{79^9GTxjQ?a'
            '-=BVS_xSjCcjCLzbbfaNL}C8Bdv}iGd(r)GCikO==*H~mDEekL`Q{|L8z0T@-2L1%fzjcR2k}q$zQ4LyRR47E$={Y1$WDDS8L$S`'
            'xiTNO`MrJG5_|ZEN5IUzdIntG>sK$Io=WcSk5B*f>#NsKkh|+OTahIa=1Sn8TF?<k??Bw3Lk(Y7>deC*SMt=8Z3YP*XAA>B&zhr<'
            'e$pw7PlY^v2}S~QUd&@;QTO_LdwalX0-ImvDelmS4|eFoD()foj4jUi_fV8o0b>Xvpi}^gR_}c^#=rEppf3)-1#xLw4VmRe$&Y}8'
            '&@zjm=~a>?C6YPWn5I=E4`@<JO`_POkOYMqgs6K3%|WAXFR=>uus?v$$-37cr1@1`^!ijinHvzkP@@@YWU#)2tR#|5_?h*g#ak2+'
            'urtV0)bZ1z$P1e#!awu<?-P(%S23)n9=x2{Yu)A-#rC1t_f%fRms6DRXtqH1gGwccO!2Je?_PgvL+R^Za<yk6ik)y4!9?oj0YlUh'
            '@O&~)6a&}Yj5o^*ka2*}&&Co7<XV_0K?@PslaGJkjGHcK^)e`BPg8FhZ;-5$n5~{+0dCk=ux3W#cJB;s4*$GF_HVB}F91e%&RN>_'
            'e6{3?P|O3_hvBt^vTgWDd;YqBDxnV+A<KFC#}eq6U<PsrfEbyX$F{QvfMhibYL?<AjY6^yPMKPXy(O^#{1CRd1o(s-y#xQw6!ddB'
            '{aZ<F<QYgO$ar3-X%ERP(G9>Mo~2oG0lkMgk-CWi*O6#+1t$pbO>sswk-$xW?f|c~h>(ys6OCvzUDVkU@gf1@(@Ue(WEKr&$pLat'
            'z5Dm>-MQZn#RSmRUL@|upkAZ;qDPSW<`T&7i}*c1^%OycV+7SM7Cjx~@zLW)m>XDVo<E?Z5unR>mK1H``D3(1%Bg82fDqATl%&M-'
            'k!C2}7}_itWhvH4mcb4?{o&C8Oxj?NfPK%0oZzzz<`8(!=b0#hwE;Ggpp)thRbDZjLh*ZG+~7z;B}tTuKhOzHu%6*TNE#`Tlwy`&'
            'WvJysHB>f*S6P)4fF}hUq6f(3!Y8RKqp<?F%am^>XOljm@&%P(^$ay)u(C1g+9K>Ds`lVz2`fJ(<r=gA=Sec3#|5hRa9###&gfq?'
            'V<TbC_?gDtb1J2{7$_z`3@+lT=kRv=<iPVly&W3#l=maok~{+u8RpDNT-xSUnb*LnBU@2!V27><FHk3hurw+_k5C!bYjDH<K3FZ`'
            'Z`c4h{&GaDE36nCt>b)2Ke@sT&@#zL;|Cxc@P~RJOyJ+I<0%pl-e}EyoLPM9xqFrx;0H-bQ`TecYI~a-W%aQjMkBZF8zvUd855E-'
            '$#e|qIkTplc1VK90)AIB4&(eYG(Rr#yi$`s=3bBc14+byiQt)*|4>cEHe$zk!Wuw_&HVW})pIq2CX^ee&@6nrIb^-<0CnIeR`t+a'
            'yXFvdc;U`*@jltjIfuvqn8&MAycw=dcD;R;bwUe=t<JPWldus@`b|F=A@ICapfzF;ZS-+GQaugQ3?4@EaNX0NVHd-9;M3hL77VUc'
            'Z?*$PCmXh)DmM1W>14B88Ng3I>#Pllg4H=|)APC--MvQ#AfW3SL;#1^u(~)PtHOA)sp7Jy%JV^JOp;u%HU?@q4f=ht1O06F_;sCO'
            'hnYbRk_=`EXNQk9d0fG9a#7T9PDf=TCJ`zG%j82~cdZ#s5tHh_=S61k9N&7LO&Qjh>XV}zrkXg}PsGyOuTK1aiW;V;^bb;Z=#zpC'
            '-`d?;7jQ85=E2W@di47Fkmo#2$|x(v5_TF+w^|XjT!4IGfZI9?&}<^bAB+`x>+NK)zn;Ff`UvROEGwO4rArv>rLWNdxa{@!Za_4T'
            'F!d&0MKl97aDpF_B8^}vqFEw(*Y-A}6*}McVK#6U1|HFD|5;Ql(}ci9qKr#GS8)+ffiv1f>m)_E14kB9!tQi$uXCq)kIP+q%`^0Q'
            'V2l){47JIY@TLGxA1$QVS+(RwqiV@N?<vd=TSxIQpQBNDJOOY;66<v*?E$u4ie6I$WUV=@(j>+o{ek%l1h|cu#KdZ8hAoFtivRex'
            'f9I)QMHg|B%NE)CS}u|`DQe|Q>c%GI+6+8|jtfLwXaPCUm`KPTOT&U0IaNK$$_m(Wt6zW&(|om}=CE@E@t7>QrC8M<=+tGbm=UOe'
            'YZ1X>OM~<rN|u6@#lZFg1A;uecqdysNZ$1=ZW#`-dWD8qLUp?jY=hUj6nU)KbFwEZs_91Ao)2W|4lu{`2_!C%uI#Ev;nZ5j?_;?&'
            '%|5hY(Kj2_I+j|=Rwam4%wYiQT;PsD$*wI#!YH3oSy(l)p7SaZyM~fsyqd%_v=iZqB_Q-P7{D+uifFw+cxpC#K*I-G_g;>EkFdT4'
            '6nva#K=mqKr814k%nyqN&0IyrOsH4L)p7qz0vL&3f#&8{AViU@#42Zj8t_c?9uILmkKeu>_Vtn(+7@O4)d3j1eTzrQVI!@;`O+ix'
            'hxk?;Q`7kC?D!lR#$aeDy(Ccjd~`0p765D*z`yk7WO#0_4!)C1F{zW3=zW!dAUZZ?t%p)1(SZ<l%B)%pQ7jC_B2F$AmHaVD>tb9c'
            '7psU~Ko9V2b(T;O0RQ65i|7I!07MVAVagr|hZRu56xVf9pzI0D!kRV$nL~GRcTN`(P!HTP=;i=$R<J&2N5ji=7GNWx#wC7z|ATU{'
            '00B8hDf)~m%+6Wp<iQ4TTB5;VTIB11)L%Hj0w^_N#YKU1D6pkeR5_uzjSFJdZplFQAvWKqjsha3@#vc7cRRf1x!n!{>89~GFOmzO'
            'V<|ATYhvG#P>+&qit3y=+;;$Cvn^0+l>3gh%<Q&jjJNFlI!%C^I-6XK%VK(dHs^mIx;;|`{_3L*FK{d)Ldel|gPdFFfu~z@U$+iU'
            'c6MkMbBCa^cc3^6=ZK+cv>Ek&J-F&0o(y|G94z49uEl6+p6C0MM$s&FD~zGn_|F%`R`e$E639cG4TXVryd&+!?C<xj%^uY=B=^Qn'
            'Uqq!bTKqf|lRQsBz^tcB*oI@^bK%sP$=6<fxly@69iA*;QGuOAGJ!_HOJG(|Q!d}Z0ZoN6`@^SFk5%#>=>_0rbS`>oG-8@2Ye7Te'
            'UQ1ssQaVk63cx;SkG{{+#n~~z(jDzHSviY<lAI3=0)SkGXAa#x1o(jvdAOr%cLKHySVMK^{uW!Z0dCAVB0oHNuyuED;ElhU{1B<_'
            'pCLapi_h@^mchF%u!1g#e-ZIUYdSEH<p_PxN_74j9Sx4|4SB0wH<+{?MpXqEJ%VpQy<u>pJ4Z)nSOE4@yFvLu^(cbBfRq6FFGnZC'
            'S<NFI!4Zu*-{A~)4ov-+d}9*&bOO+g<uLZDJNX|q@t-j1*7VW5|JK21n_-<vR?lF~<%u5^7v&Iz%jyh%gY5iOh^*}et(I+mQ5;BK'
            'p$I9vGcMz)+|lA+nVBEaKfJ%S^WoqzpaAep@$4sl(d_#gHoe^Ov>gvN;$Q)o1Eej>fFK^EsLj5NQxPN41YQR_JrJ*vnwFLn)z}<R'
            'sf{}Q8eLS&3J7Z<dKJvXG=e_#k<G~R?BGLO<fx0Sr0V_G1rXxzU%mN3yi>FAPV|5z<&!clfZ8+k!4wE?1#&*|YVX=p@(%jQOw0u^'
            'i^LogCl370RVLmYG{9lj<7kB(KdKIx?S%a@i)RM_C(mZ(L75-SBVgc5iI{6LCW`1HLivS;y37!+<a|^|2I38}OaM9p!7-~*7Fxlc'
            'f;mA_8cLa{BF7gH5LYlYL$wqj7*#O4i~I`Y9uRFL?1U&F`MXv)F5*?812hFpc_sT5XBRM2&^Y&P63@{ahSw!c)qKNjLIpT(P|b-S'
            'HAR=^4xkq7EXzU>j2qSvgr^HM$bqB>h*50RNMbm)mINQFS1~lYh%V!j?IUP?DP=!eyq#JQY=y8G5DllMy|nnn^Y-^6anz9R=?cxJ'
            'XQp@*9FWhciQ4pAVK@FD1c!&O5W~;+8hubcMM<LBO@PQfXd_fOKIU1M#s%d24Z}h+#OZ*^8sZl!D2|3VX=LVslC+h64@4{2FXGw('
            'M~WAE$Gh$900{Td@Z3~KNt%kKqz&aaO_SO8KyN85fNto+W2%?sh{>frskDC3GSRJGdo{VsYg97Y^Zpa*9ckuan5bhkKR*BIX()+J'
            '>-}zCO0O93Q1u)=vqr~#=}so87n<*2fikk)k?|0>6+3eeCFOJ|WkQ>ut?cPWf{Rz{YJ<&GC~Qmd3-Ae03CO?->cm-IUo1!{PGBeD'
            'uEYtYy*`81(j4_0Qb_w!P6UxJppPJQ<yQk~0b}|*hTa%w46gY3o-}#Q#h6BA>JM9bU=z?FT4S_8j}}Mbn$)^VKZZ_S<`8?E<$nPg'
            '_WKu4kB*M5M%GbT%3jel7i5r>uJVI7^wdKqpST)vpMh;A#~ss#$5o+UQBg^yiZvY6W>6GGk_kNom_BE~ix~JC>;bd}98u9z*o`K1'
            '+@<4WK86KDKf-th$W2GCtE81++L~GU!BNOAW5A1XC@xVzfwn1-LwV+`VL-%X^ny{)Y6+?;7o1)|TnSK+aZQ_1|6G-UM@x*JA4yin'
            'H=6=h35Y(W-nP_Ducp0`u)3RT-h8Q!JVgFd1TPOC1?F)zPYO6?Nu;x&eJS1<Lwko?jeE$)HqM0vJd4XU9!Cfb)zlB^ymNrT6qbPL'
            'MX{lBCEB{{EUBbEGsd1vXHR&<C$g6hY&QsIz*exKh^mEZ(hkM(80aznH#o8<O`(=#YDkvs&By|N+z$o+meBA8c4`!qS+J9)tpH+l'
            '5s&793?sTmki)^ze0z9J(CAr|nONqt6n0tbprBJ~B)7>`l&B3QSNVJ3z?;3@9F|m^-Ph0;q~A?f49Lo4bf-l=3_|{f3q5_m#!wX>'
            'tR)akXIHhluTNU(*N`D`cKgp%VK7Yros4>lzWh1I6_$Ozr&d&+6eEjZqM8tCf$@q=ra&qiH@gm|;G-A<hZpd(0bXP?kn1C{t&L?3'
            'YVAfd$&2ieQ23Z{Muj$Vb-XLlO6k0ri!XbI4qOUYTG{P*TC%bOO$2bDzmxR@vb$Fbn*24#^SgQ{{@3ZNmvnw`=omush(NfWjwxXt'
            'rR7G*a5!R)*?;Fh{_WpobXrNuRhFJ2^jp#%?r70a_6%P51C)Gc+PDE-tkW5C=C(<MaSP-Vlxp%hHAL<}7^|m(^jzhNIeoj3P#GhD'
            'i%Dy?2M!tp^YF?V*qPA;vWJ%@+Z|Zb-@M=zpx^l%|G~gJ!Ud_?KW~t8H$xt>a0e4H3hHWp@BpAt>*Xkr<^hT5{k>bU?HxN*&W_rm'
            'E^B^b?Mb;6a8Lt^f$XWmFbIX|JZqOpq^w2?Guk=@Kgxd(BIZFnpQ9Oza74&XZajuE!S<SWa=V3u_tKunVlY_)P`XoxEL(h_gN~)E'
            'PpmGQ{$gkcP6VN+hm40(67{rAJT+955^a=TMVpee1!bc=z)e(1F<aSt2_OZPBFl^t<*PRWk&49jlC~e<$UwY8Eu|LD;knLqlnD96'
            '^|G8D$e&2*WEe2}h=6fStUZw7wb;kB&p1)aacQ+Gtp*0|uAviYLe7f7+pS<N^o$URvE3!E1GOE7;_Ny=6r7H$&6@qRKuF8kSA9mt'
            '<|M2BRXWeHs@{0F4aHzEP+KLCn+i}fP@)%`qq*>wioHQ<fw90Sny(8q)yHf*g+6J!tz3KR#38q<C@bPa5nsgb`LIP>{-Rz*Srd)J'
            'b~nTkXr~D@)$zlNs63!?9^w;ed=K}uUobh~9;yK_V=eU*fY`vICg_kV!=r(XcJ&TS_4Ee1f#~tUt7+cDhCK5;%p!JVh*rraFK$l~'
            'P4U|myps+>_oa#tCvTOK7;BcBRv@!rm@b|fQa}?w{_z(X7d<C09&!>TZr1>)%(a4zwt~C_GT0&MdJ#?Hs(gr|ud>6wd#B0jyLTc-'
            'v-*|ds+>S94r>yRUY<S|L`2s>Aw-W%%kLvR_QiC4d~)wDEA()Kvp8!2V_A@-<XIB4hXL!TXxE~C8{?f3rO+oRL>uQNosi6xk=$*-'
            '!-iH7YW@Mx0<gJS0Ky=mi4?V1yzLP22WqDE!~&-N{N~xggNFt@d3=zO1Ti_9qyB3e)g=oav>^&MjAU^P0?MP)$IqWb9|{l!%fVmi'
            '1k*gJaR9NPE`@ESb|JTU3UC;daa2qfy`rIdZ_BTGZ)ackht1!;?}l#&`0KmA!ImLNnnW+WQzn;RLbW0MclN*EX6Ijh_Y0OE@(%|T'
            '+t6+g>Jb@)TA&(Jk2P4OYb2*|qtoVlvj_71?f7$-H00$b+9V=h)*B)o28(`tPq71X$(a2lJ&o+S(LK{ZReI;9_ucQmcsuLEDqWw1'
            '_qP4_`l<IFet?-V;0@4xx$YeqUUNjr0Vuk`=|*I^<9?rI7S%QhGdM50x;z5>^-pfDFDF2$nr=Mx=wI{5lV)no!x@V>ylpE~C)oC#'
            'ZNkx8y1Ol;YTyH7UX-W+$m%MF2c?%7IW>4*N>j?T8$;@kGf4pZ;wy3I{?U>6nr2r*$G7+K9jj(!Q+?Ta?U&ZD67nhR;seYsY_}E<'
            '$^u4<>)auFRW(rfGmhXu4W#Btf5IAYl3>Yqa9kiURy@TOnu2ryN7x)NwwD*5ux(GTf#$qm?&kTJTc-$%Y}QBUMXY`*(~kDBA_D2~'
            '*@4&M#q=DofQ(j3DXYd1Rp^6Rp?jJ+n7%gKXd;0uDe<xqAWT_0PHB==hJzT%68XVpXn2tX2*(nlwwb;FQ+aU_7Z!X|(DMp;E!~WE'
            'FzXbs<pPJzv~_EKTbf={`$eSPE|j*LI&8|gdUi;uy=R*+Zj^@1{@E)j(b?q=Z;8@j<5z0>Yw4G!<Cb4T2b<Pwzy-;2a1SU(^uFUW'
            'Zv4TKN1aiVgi*#AgaoWdmQ)+X3oA2aBpqCEUeU$cl}t$5ig+L33dOTPI;TV;U|0c{geK7a8OU&ro*(!Ldn5}3W(PB1l#;Y$gn<1w'
            'O>6QVWG_u69P{40ikM#?-mv}pQlKJ{gt1>oJF!g`#fK^K*GxND$i>YMJSXZ%kk`kPSjN^$IT1G<23+Lm>!|jv+Mdo)PP-H^l{it>'
            'O(SV)Y@obN4m(8?DWy`>DD<0hwD6RvDTe~|)|OOw{y7Q`Q{QZ*`G%&sc^yc6<*Ai{R9i_R=|rcX9oiRr2H>-RoT=HBJU|PP9b?lj'
            '3i3BG)SkjL9(skAza9n`Q?3K?tvK2<<t}gAh7FfKtB}HLJ3k4!0WlB5Q?3)<@Fza-oliCdBYtz57pM<2U>MskE0akY8{c@2mHgU<'
            'zU{ZevgGPEK;(uR%29^Ro*AlTl~Gp~Wkzh)rx-K!o)g)kT#^hr(|YC(1J@Ddi-rl|UYt2Nh2`Y)-mL-_cC<&6(JI(l+TBI1rS2~3'
            'S?=z_?d5Nmb>@e4nCFsRgBkd|SkTA>hyolR)96Xh&P~b7kRM7?qmU%8-ooZ9xxT^gQe>QDtj`U}>ytq-_k6y0<HDlrwbkuD(TN*Z'
            '8S7bn>|6B30@)zrRVa`OT3^qO&Up|lPV~c$&r-VRw=tl>E$Z4lxW3xHzg`$o?Kfay@5+iyWbq5h=)!u>J?ML)?rw$UmLOvaG4oYr'
            'A(qy+UpFEYl;MwBNZ92uV6?&#2~=;kRdC+2^7Y;T?hr>E8s_7ECsf^cy$B}9M1xi@_HN$U8ZfzQRE_lSa%$*iqzl8d%c{vhS=#3S'
            'H^M+vV2gT}ecC;jxO?QeM&|QU#j3Sw1V7xGNbG$n4nE%>fOe{sffsQW%Mtf(k#kb<POAu0u<U)Q{lWr}oPK^ee*N_2lc%r68g_LI'
            'Jl*H*HX8*AeaB_N&H=!Mtky~-_If`zk~yqr4>;_wB}W?qeGy~Qof)}@02jQ<XR2IC3_2G!#s0D>sVj#dud9-2nv_agLiX1lRXU(I'
            '_}EAJ{`lzV$i@~RG>u+hcyuz`4zEq6fboi;(C_e?8yp_p!Mq!$$NCfya*R59KxOjwE_>=h`VS$adCn3FC67|kD0fe2Mw>EPG-&{w'
            'q5dH%fUX3{NQ43EGy`ds5WP9!%K?Z8<r;bWY6F7uK%AoUvkU>sHbR0E<b^B4xO8vWZ+um!Rk9{6_9{lVG?Wbhdr?+_0cuege>6Ha'
            'I@~BEP3SJWQ9tnEjC*~q{BleOKk`?^6hC`A{cEgvBS}MeGVOZUSFM}knHzE!_{TX~PKnE>m&BZ59whAFkUFyM@9qiZ1(VTC1M)IA'
            '1_VII2EoOh(X<1&Y4-#KSChr8vt`Ktj8pRAJHs+|fiYE9(<&kD*>29ftlYYuG;~bZGX_=GD@;`&iym3NUMkfM4b^q>f=;Lz#_#?1'
            'XX^de%xzE+@wGU1C)I3!XIXLyQ&7L{TzR&XMQAf}pdXsk;_)g&*en>Ba|->i7V|5D-mdONIAf_azUhV<bPGhlPBEPN#lgJxnrjgL'
            'a=Va6&Ejpgi?8>--Qw@QB(n%w(n*dvhm)m;FKwcz)t!xvfbGH-jY($jLxhM(8E3Gz1trX>>8QYdj*bnkhXANyksf&V-q6!qpXdA6'
            '-jfetYgFiO;kf-)>?lYR)8FoQG(u&V1a_2A9-xh;K}PChtzPgplQT_~`fYQTa(GY)R3|aTuIf<ICQe8S1+0@(IvOX*nsHW**BHE2'
            '$IY2_gHy1OXVDmA!BWy;Xuv9UaT;Y8b#x&=SP>XEK~J)z!fXJX8`j7Rr_atedJ0P68#U|foP(2S&w*Z5J~`9mBEzr?I3DS!Z%n<k'
            'Z^WyNPFLv|A$n2=EmUgo>4ryxLPd?)Iw{gi#MN}rccQN89HESy){@ZKtjyv1L_RRoW3cf`5uvwaDLpmih=Y>#9J7Y?23C(k4jLsm'
            'Y#-?$QYgK@+*ER;V~(dc;)5-}VDwDjnH;N`lri7G^3NX;37SMEgcGn{=GR^uVK8^hWipLN!E{|yD!|pc#?1iY&M1I6c@S7fE-@Zs'
            'Ou!Y2G3hQx+|;1i4|YHrQUxD!z}BL4Xc`k#>Z8kOCZiHZb{bE4oXyD1cuZM~q3?NJ(gKd8(n4YUNIisN0&+5qc685%W~2D1Xa7bu'
            'FZoUTQp~BjK<a6*KLdxB53EzD91z=MXe!JZJ$IS4Hlx?+`eE;}NnSF2T4I7{^!f$NoYn16iyQfB4}Btwh1K2x?O!Y(+)n4_CNQyk'
            'zeqkrZq^vlqpThMhxYc=n(t+u!g?oZG>w)6W1JmIM>0rvB&_Jx8Z>Ckdd~4jDmuN}preJBhQe;Vtq5_oABerjmNWNmtCzQF1khr~'
            'dp6;e*&!u=jz7<W9;X&ZkCZGCn|#?fSZB%fpvrwD?pxr;lFY)UqSnrmVFK)7Q{$YO6x=%n26^?4Le0?PY2B3U8}U(yxRg9_<QwLp'
            'X5Wo2I9$Ax#jF`?wtWITt8PSZ{vLMyxbJ-H;OOkgRUBj3Bs^>wF|&ud%DKC(`D6*aTP5)GFSAV+t&*uqbXfLo!@S(UL8vsLpUmLg'
            'md&{pnBTw(-IgI@lO+pv&ABGln9Cd9v5}l`30GK7xy~TV`YMbjQkKM+K_M5<X+=uus&+u{7s<sUMU^YvK=R6JAObdtu|e7qqq3E^'
            'S5r^2M-oU)$tj#N(=GwFx$p0YHTk<_NX}a{+ju24NxaAo3*uz&lb_mht5QGGcjkHen(o;j(VgXn{iWF>S;eCpGIb>yG!G6BoO5q_'
            'F`Zur3sPx)SUK7;kCsfx7I}1L+%LV2rRTQgZ+Bzd8e<)Ch|D+eTs7_VL-Q5Rov)8r_xB@)<ZoSSS*Yt@3pm9ev@_@7?CGWTLecBZ'
            'YG2v&JIzw_d*X5LbgZ(e@RQc*2(y}08(rTRvGEzF8u^z{-1UIo_Wu*PWqJI661VJj)1&E=r#7mQ<jL*u*Al*QhC5XWXjg3#xPK#y'
            'jz`NX0X{oU@h+&{Ec&eeAWmPu5jS$@QUX>P2s>3O)@*e$CZ5^1@uq#oj6BEAo{woTh_^$d?Vmo%+*)29uu7sGmE|q%KvKC1i<mej'
            'q0lhS*rp*i%=B+G5w&5nwWwt;^3=7AMydi@bVA{Dwq4;^m2YfD)XDAeS~e-IL<o-_2iGQ0dKH*+$ywQYnpL&uP{sAZsX)qlH|yH}'
            '`@jDuFkW8cyPo-Gi*Neoo40f|gV35m@|V>a!)a@>e9|^j$=8w}?mi}42CX|syZrD@%&VT<Ka(Pkmg9?P?Yj2ezt<SanvOQ6vOnOs'
            'l&(&47G%*X7kNRhv+N1GUcjN)1kfn<b{gS%qgY6GD#jgAP~LCWq8p0n4$BB?*SU(D0|59i@|Kk6z80{~fU#LAr7(+Dm^=_I-3$l~'
            '5F~D@Lg)S6&H`MPHPPZgzLdbuj?Noxwgm=e+<~BqwO||^pDXxID7NG;;aiC=0u^SjLUV;#QCa{C6`GHQQ4`zi1p~y?FYvGL@UJ2M'
            '^)PVq(Q?S<nD9j3bYc-j%yGdj_NASucg!S6BXz&cxL|4ntZA(jy2W7q2TgrKbbo!l6yHiy_&vohxtqaXAJ^bpWkx?%b*&xYujrro'
            'j%zt2o|^O6=F)frG<FM~PLI{A*&ln(%eHaQw>x$;B#9xHuuMquq9+oFfT;TD7oWhG4=Rl|n3aG0UQwsxqX$R3cz#`Dqig)`1iczm'
            'qFyar`)@yN|Is&ADS+RcAc5qoqI6EmlbyZt5M(dm-2uYAA4`wJdr6juHe0u5S+!eNXSM(68?ZP(pK`3)RmHv;%RCJ{Zrr~jYKTXt'
            'r|40Pq9#WvP>2D=jx%my9^Q4nR4N6`o*bu?^@RXEjWDPN0ia)W98prwm=6)_upn0*&gH}w7rrN!a;nya$mf!i<=jmOgG31Sz>Y5S'
            'WX9F@)#jCF7LeUdqdSRb`y!st<0;3t=krEWvy6&}?t_gkIN|FBt6;?AX#KdZ4k$`?l~WX-3?ZZ*Sh^D4T$rk*i!Cr%fsuYPP;b-_'
            'MbkZ*0lUFoM)t_yiDcSw#S&Ow#l3$3&OH;eXmx=M{3c(j$gTzF=rI+Y!PzORa-V1O{M@{2i{GPP+)!NJ9UL8fZG4>vKfxY<9hd(y'
            '6y)I35Q+ZQSME=4xs6p0T8q+n_B~8ZTs+EVr;BKoUkw4XtXolkEu!@)dgctpiTU1qKKteF*MbU_)_9(gGJPmsMtHd`UW}6IC)P*('
            'R3t!BHjfGaLUBqQsPW=eT_J+1pN*6jd=D~;di{N#g1Gz~*zIH*il31U###Um&g*3iK;RKa`!Dj}Kf%AeKSS{d14?bsG$k}1-2<pV'
            '^8@9T*calpf<O&lsE60;;g?W6l{Ri`VdOkMBE<d`S_!W;ecpz&Seokg;e}1R;Dt*Jw_#v_XNrh_!hinZn+Njo;5a;@|AAaU|6TJD'
            'ZPTg#@M4Q)!h7)Fi3Ju89{4}InMv{wojdMChNncyyom53+MtAT%36=7sN|5-%qX&JLW!V+;!UjYmE+S}5>CtIf(Zl4L+?ZWVe9;P'
            'qxbD3QL?ykPJ1f3w{b}47M@b0Y6r)f%()pfV~F1q^X3D~mhSh}m<BhtTcHYP8cY{+41{TJPK4WJ`O$5bXAZdYZq~egsaFSlD-2Z8'
            'Vz-##q}vAirFcnB0f5k=y2@?o2#z+2GZ`q|xRQbZ_b=fn_dvrm`J85WElN7j(#=ETdZj;O$eod`3VX{~=m7?Dd44Gm+j)cxk2*RJ'
            'IIF~TTCeJqS@1H36CUr>!@H&idwvZhW}5P#o0OecGa0t0@Blv?f1>Wwt9kV=v8jaZ?x3Tr<0!SUN40Lsu4+TyZfj}-sH3&k&P)Sh'
            '?s^oPtob~SO>UKKtDK{`q!A-aN1|Kw&U7;h^zx?%|AgC9q9lz@u5%t-U%!1jZJ!VS{PykSc)PvU-Q5N^0&z|q3Bq5mJxV`TbdO2*'
            '%%LBIx~}5E2H~mEJyrAlCrOm%7icVn)6CTolBHnVdXMIUZRopl(m2HB`5Im~o%Y@@d6R#f<Q2b`L+R691E>jWJIwj7I+Xk{=J^VS'
            'CZ$K~0`<-OCR^#a@0WrY^b3U0y7Je{N7(ln8W*>CE}>X_^d*M#+uUC`zv=42Mfa_R3-iLlv*GdmbLTb|83Mry=iVZ(#;csIeV8Xi'
            '-C9`2DF^WINBx>=$&e_zqyrC{BALZL+xJfiM)S&hPB1qSDhA_t$tpsXEeE$NC1W32*J&%jp^hB_apuZsI^>s87NwgqDaQ!H{J_^~'
            'lYN5{pYCaOS$xG8vNiGc(?wp!nXGL@B&{fk8}?aw<kGX!DuwD6K9YOrE*}+f{|v_aGOwPY{#|ET%g9_fms!=M&a32`{e%qrnI+{?'
            '92{`gvlSoJAYV??I4YRA%PH$Sr%(x{Lt!K%i_Pq7%uPm<BB2b{hWmk5xytp#EHQ}64l>A?J-*$C+<0UZue}g>HZ^d#l3hY>`1Vry'
            's<ultSw>(9D5h!TBB<Hbu`Vau1&qH%Em=zSm37+V8{2FZbO7AYxKm74rkt;M8;MG-3M8Ttz&!&Eo5=x+(Hc{z=D?tqzOIMuaT7SV'
            'm!Ug63HN;H+BTDtZcNI6*3d{o46d)Fnzi4*;hyFBK1kF_nxLvB62Udi$u<D)Ws{Uq3R~|YNr&Q^Kz~te2PO^=Fg+cd10U+8K!z0H'
            '6oY+BVbW8p^Q|#|JM<^>eS2cjz{yYBzGy8<1>A7&TL0c+&ae%1hI;Mx#Ko6yJxj5(XD)!=wetqrZ?8D_vGu;)SJm0>BY^Z_#3OW<'
            '>HDsQ+(msG3gLl9sg1AHosO8I$0kg=L-zYm>4basI1PJ9n{K@Q?Nd77JLha#X>`X+YRt1Q=o_nL7e2EKo`2Gk?FLCsAN0a^A1C8>'
            '>j1S`;^1YQAISXNK!yo4khjRQ<OAM+D3-vmC3MnODj_IPhgmWZwv-U;Kz7_1<ucBaED@MeTSS`(MD9UgnERV-xy}TQ7=Mh+PF*a9'
            'a@oO6si`=|vZ<FwY`11ZD55(rl0Vb62+L@bMN63|&fN+?JdL&7iZkhYVUVo@P2<XQbFX;@W#i-RhS#D!t-}R|+bYyCZ1?6H>`?MS'
            'E_ly}9Umejf!8fIgo+r4`nC-4I>lvD;I)rQ2{Y&1EfY91sm$UY>Uz+k&1z~JY0KUeyJnEWe?NWB=0SUJ^vn&bCJA^)++fWd075RA'
            'qyN*ZKf0(CSed*&mBTjsYFtiZ->0Pgf^=ke&&X-!mvH#qm?;&d`sep%dFrj<q-xjDgCT)xmK6F~-;XE!cc?Wm`E~#!E0Z#nsxL-r'
            'CS?=dvuyvcM%<jV@3Dibo{+gTRWz6IHT88N*DCtEqtN_#3ukRz`XfBEbsgIcn^JL9d0;0<=EQDfeo%!Bdo)`UJyqy_cn7Zj=-8G!'
            '{J*Wz%~ak{+Hdx~p_0$mdqWk+LT#6}N*Olm8)2I0&{6m{ssQI=C|^=@avP)D3na^+eE)d2u_gtxtG;^u{14AxK6=5<e#E_KZ$LB8'
            'R#GbQ8b}&m@Q8k82Z}B<7ldk)?o6uHNC#04#B)}I&+?d+_LTk{DCCP5KcWcy=FKUN3ddax$RXeDIbAG=cf!^c-vklL^wV*32@4M*'
            'DGdiltZ(MC)v*HXx6gO4%cOIAraqDy<gVzekQlzq^~}C?QitFAY}=+eFByj~$?J{mtB!QC5mK^@U)E{*ll<J191pqw50uA0V~9F>'
            'r6jLuC?|r|32s!efzEX2Qr%uBJq}y69trYa>ZFLrl2jVr`da#O`?6EW`jN2`4%1|n(A~-fCrfvsY?a8QSz%b{CB+}uy_Px%n_;HH'
            'QWu~GG}$N+*NUsAQs+apS199ouDtl&$v6v)H^KSV9>40KypqLfbjO@H3>V4kF@<6+N7okYZCf};7X)A(H~G@Nd>f`ZzQi4bHPfiT'
            'DW*%zdQR8+TlXG5XD;=e>ndkfx6b#B-9(qV%bat~o4!+Guv)_3y>*1XEh;92w0PruY259@n+e_cFJh9vD4l#lHvt5HNPq&RK>cR0'
            'dunus-sFYS|9oPM;wN)ilA7C$p<iX($yv3u#=G>z<|d>=dW<$Y*H6>Td}}Ic`<!6B-D^X?MFo6vup+tZDo__d-J%v=?W%$(bq#4)'
            'v$zd(RjJd;m_O@t&-xYGgSWxS+v?@*EN%mo#tpv*I_2^XE=+Cg)1SY8^Zd<=r{iCLcz*iR)7Rr4Up;yH;`G)VJiv?L>3&(hf>s|`'
            '$~;X?QmW|g&U{g9A3CD;JZNsMRXe?-ul;S%x_R?2HFj$@Zt;}TaN2&4%Sim74*eMb3#o6oA^~>?RK-mW&D?5?HBQrht$Krmy1d3M'
            'Z-?~#0hb-aY_~6C!@3#nv5vEFlq*_L_H~moNVgi}3xd>I<J^q75QQF_YJ<TXcm~QN-7A8#=-|Wo^$EJr@0joY@OE>s1H^4+AyC$*'
            'Znad`#@KInhY>_v#+yha%R~VGJx>!cffGh;7B~R#lA&7UFd^{GJp+4rk;Tg)##Z5xQNj7}5as$T%0}0%7GYbtXJNB`lAvJBAmgP&'
            'Wn8pu%D3=x&sqm9MHzIed51<$yP>l{e+|w(pkx#{x@)yWuWoSoRrxt+c%~3$^v;#dyinajZeJT&&46qRa9+4}5f`c!=bRa6-uqzc'
            'udNwq5{<Rt@%t`#^wk58MZ*?2`PvVSKem81PAele*LzX2G3tlxyxe!d*am#<yyFGt_yC0D)oh=-gie^r5I0Jk;oZ0P2~H(?z9ZRG'
            '28lyKm{|sm9@tXYG)WlfmZmXX2U3OV=d_sXXS#+S=@Yb3L>S8lJ;!*Tqrp87rTPefC8<rs@dNlz=5>%mP!}p-V8jVWVwlP*@s$Ou'
            'FAfjI3A{Y9cx8R9LK}+w)D32@@qRbV#r6d2&x6(aZea4<n8L9aC}~7W0?W8eQiIL*UUMCquFeN^J1jGWih1svcih-aX>d@P#T9>T'
            '`luYNqBYX;qv7T7h_1`gFQVbmcc$0r!E{KMjit%t{mK2iRG1ePA2EZsL<J~W&@E#p3~`;jAJ13xWm8|?JvmxCU?cn#MN`>XdzKY%'
            'YiMtHQbW%z#v^;6&7$diC}sSyNl~2cu&Yf}I!a%%iEVmPiLooEFUPmbTeGvKd!C%cmQA+Z=I+~;6t+o&%MwOVp?atk(K2DT`iDZ9'
            'We(_lT5X~a2{O$!Dw>iqT16iQie%VXRYv00@{Ds*n^LG&njs3jQZ}Lzxu#6V^vZt8ZDNHIJISfqF}kg^A$xWxOJjO*LUp?}&JFT<'
            ';&pe(8BIMMwA;--hI<_;7koUCqeCJ+A*R>ci68E4tJTNcdT`2A2^wIUdkOVQQPMjv{cAL!P=x89o$oL7K(VlbFV*DmJxXq8Jn}`F'
            '3Ku*vL(N^9*SgwP532-{P$cL9VWZ5pukag4ScN*;@kJ4>7do6?zvdJh^s#4{RzmBH3hl$jojdf0+&y;+V+Y8OvF_JKVdWoXUZG>z'
            'B$(E;?GdD<cA`}~VEbe2_FCB)LeJ3bm1)9iM&^NS1m$f;))^e`wZi6*IUlt-{IChC8?~8s8;;`oMlIga3~p*OzAh9M@ziE%?UF9P'
            'KJrQdb{(4+A9;9tyS!#aM{OR~NfKc3nWGL~)tj`Xp75$lk8C_O`H9~miRFFKYh!|u<t!(CLsGa>8F98MSC}jlc<PPwE}Qz0%0Q7('
            'aNW@}4W+*g_9+*;?v8npaBJjDCKNT8@1M~f+Ou<9o#47PHQV6Y?eBJo_f_JYqx5}tnb+0%e!B@ylSVc{wdu{xNgGhd1Df|tI9B;j'
            'SuC3lYcP_MxYHeAu-Pf@xa*ctcN<#qXQL)4G_(TGMn+_nt9CPj#YT5jwb4l^1DA|e)&AXn<l^a(`<D>ViQVR~2eJhH*ao*)K7H=('
            'T)yTW3|@`#lx}W*j}RUzWwU)6h0c)96;DRoAm4Q8KUlZ~1+TWm63K)y>kAy%Jb02UV~i`E715u?*ZT0R;w74w5?~a_$<UDK==`K('
            'hz)GWVW4k0x&1eevPk)G?DTqofAwxH^&}mF0erG25QkvB@JKqmiY-y+){>4R&MIMtE-lfbZU1BC9_UPdIZvYtVuZg&NlNB=C&;a3'
            '{>@U6M3NL;X{H<L6KpPy^6H|6+w)+Jw24al(c@%U5uRiT`L`Aza(GNhve~9C+FG4!#BR!s8Md*0Cl6-$%s97EjE)DMG$kWW)oyw<'
            'eI-spag6uWo2+(gFlTB~Xwvv*4ERB_n&nq?;r$0-+Xi(tMMuV*DMzmp99{rb^Mz7@)YeEF@+@GZK&y(p|KeO8ZJ3c5e(r8yEX_&h'
            '7QqkXYA%7k7dc+Djx?Z39!lrE%UIKgBr79~wVTE$G+@%Dc*&{_(*yga-Zm|xR+==`vuF{4WcDGA6mIcV#wf1Je$_^Xr#zk2!Jgsn'
            'Xrt^3aB&{9l0rIrs8^^E|Bw_mwnl68t-!QHDv2IyE{zbOmJ=4&p!-KUOzXU&QX!y9hvuYD7vw5p%+pw3YO-SrRj3e)>bUB5x6eoe'
            'j>-pSo^uNI@eFZ*>1Rrg<U>(1U!EIh4G-Y+AS9iPO-8lE$zf`wU6U@uO=`A&znz4~#5KdyzXO2d`)|ZRM{xF{CC2IA#@Zb+<6OP!'
            'wMWs1gIA|(wZCApxIH(9(VhbPK36Uc@*A$uJi%PzvS&n_I{IYx`uC`qy+_k!zLvA*^v@BiKj%0)!tCUZ7+Uxhy5ALv8oI{8+)!fx'
            'NdR_YRvV-VP9i25Kuj+3RSg8<p`TpEnN6@hmS|bMvGi#&Z>pkT@>pC&Gw1|5?Vy^);9`xj!}bH24iI!kyWh_486+8;e>ijQ-F}YK'
            'C|_6Gb6-123j?o5sjTb$<^u4#4GnFVp3v*k?l^}Kp|jmZywwa>3v8$L(zsL5m^MZ7+7s8%K=|gNj7~3FR2Grgedq53?d6Mh50q!1'
            'nuegmwb)HI6c`Gr=9uuzoN4sIOR|igPDxW;gXu9%?TxpjzCC(}J|{5ww5Fpzs1^^`6!k*gp>;*XxQ$NMhNG?V+J;X(=(OkG0uJk3'
            '6l!075)>KzIP7#A=z?G?k;gT26J+M}DU{uvZRr>BS{U<pM_FZxZ067JH`L53y;qD(Cp<lmUEt*G-WnQH8GXL&0Be=p4O!ZGQF~k_'
            '7F=zKc8hcYY#bDZ9zTa&l_dV(q#U&?Nvgj~O(%o4b~ddrwF{04<m~{`a(EbuTc}V2zw3ipxl5ZGw458vS{(}Epyk2fR`TeE1J_T%'
            't|Qgb{wiipft$3xx>76Ez>NxE#mksS4{1D#OAjN$hX8ODwRt<M{n=8RKp9(&;w_7cm7(b`;5|v36R~xewyC9;NKWh6%H+WICc6w@'
            'YhV<T_c=HX7L~~~d6cY91*Iu(RnRJQ(~+`yZCn<uxonp?;Tx)ONl5X_$EP_NT*9g5S6bptncffXcnpgX?J;Cwl=%rA_53Q)t}*zm'
            'Q$|<J*KS8TL^$N@IzBlNIwdYdfUa*X22q)m8>9HpF*JLTnpInEjAjwZNM$hGsUGy!lSxF5?_*^np-3ADnM&NZw?I8ZGw9J+_}M%|'
            '5vkLzw?BpAr0<`xMZJoY`o+iE8;Tiqd9->po^0ez@8-*>QZH>l=*m-i%9GDD9(**Vt-2QT4;)0Lnz)^U(9Q#cI&<gp*qd{wF#fe('
            'U*^d)Mn8D@r8^gAD;Cj(?Y`BZJ$7MFE~AeNTI>#1>~5z!ig8!;;;f!7;dC6wl<8z<f9-C}IX;Q+xC9;&1N@pSrg!d{*Dkr=Q9s??'
            '0(Ok)PE+H-?r6)MQI_2;-75U@c0qXEjoc8O_X8~2K^k9T8WOx)FO3giA4q%wHlZpuVx41e^d-wwb>~IPM@Qog5Vv{~lau%=l8hPN'
            'Pr${119=bRdn)hHwSoh6LLVd6E>!nD$7OSHa}XNVg{`0NS=U*qy8w%rER-b%)hXkj_*zUTCr|_FYq3h+qjHI3_Bc%sM!>PPgs#1Y'
            '&7ri&=|<aEIOL8=c}SyIH~rZ$d}E~(=cp(oY&d>e4H?_WDFRtDMNYfxdXgsP0{fM1DoX6vGnH2qh9oZ->`NezB(c^l3^~ovyv|6+'
            'BpyF{`SR5p#0$W5Oe2Siz)?fWF)>k6!LDA$yf|5|wKf9KyC$zR@d#!=E*CHpI!2KV7?_2ZuRLt&vI$%d-PfePny(<z(<%%L7M4JT'
            '=p>#-z-~b=Ci!fG{(9mH;|id?grhh&ApD4*02Oko2n%ftW<j~uO3qds%>e<gDCOd15>e%MX^vI_jl^oC@&OPSaApqE*z3OwWu7ZJ'
            '9$xv=A08dR(WLSc<13yA&V~g)=@3QVm6{L-U38%Hk6u0z5#bdst8WU7`=IKzbZyiN0hHCSL#T__)VM>TPl@IoyJM;M2bk1MP>6;c'
            '3B%}9ENWRLX)0pOxkGuS7_X=3M=%pLt~bw~p01`d@5G_;zNNW)hZIe2cnXz^Eu#>$9Ip`t)rIXmd+=f!QSwQh`T1%QQPNG=dT5fE'
            '(G6N=nCq9~)mq-Loli)}Dmc_w8m{_>(lf3<0Jef}gD+`C&_F}Gjj}(-l?10+#%}54(vx@*;ZyODW0GNrc^VTKN*axOaEUjEH$lB$'
            'D)I_5j@dMEWMoF7S;4-cF1?DQ8Rex$%q<uB6?I6I>m&n8IQVRly#MWkBH2#@wf34*9m6KiV7`n*on{m)W4dEH{K7OH8ab}~fSc=$'
            'U8qFf6;-|7+2%G7J=G*<Ve0SqTfNm=_H%c*0o>8&j+Xu1w$ESlKHyR3SeinC<!9>!gEm(9MkwmNQ`48ks4LMBMlqKuOh{6|#+&8k'
            'hHf>YJVC2yhQW=#HZcEKE9zSsX5{5_{uQ^&TAB8I2A+0)CBvKD2pn(cf#T||JIY#-ZGOC!!P;(r2h=3RaPN0VtQ*1V9H;(CA?oHZ'
            'eru$<cZ^Qhy3yWWD|&tK8^YI(?#U>2y7Fk1lpJ0;Phw0gU%{bMpkaPQDU_QjihzaMg-ukyA9~Y?@i)^T9ho=cP5f8kNn>>5k7vA$'
            'CZ$RY-Q0gO>Em@UdIWn0g6VOx6T_VFYEIVy`Vu6z@;qXnjk)UUv!DfjcSb-D9tE>#nNXGp%3Ww)uK?JPcu)!0f_hVzs~XO`p~~=J'
            'bfF<F|NUb8{}KN7Hwu6IV-tuZrpXd8?;p``e`nEew+MW@Ip&Se9o8tKid@~bSXwb>7SGZYW`5;!N2Wc>FyYY(9d?vc4~Iw@2z<+c'
            'wC2)CAJ-Oo2fiyB?H|!upCLNyw}wr0giQPu!zF%0sDzC%f3*+^MrwPt_?@{_ZCcV&xQB+KJKUjGZ!6OKzb(ijG{^9t9C7hA`7{2e'
            'xQpuO^T%7LPhP3kJl8%)V8&U*Z~lu$W>{I8{x%UB)zLp9GyVrhX2_{O<mEVMuCvw=8wrPOh#ogSs!;Ly5U8<u{_^zEOY!REqdz?t'
            'PoDna`HLsdUt9OKk){cFy%Z$r59l4e7I_wJ0BbNg5XNbK_SlXa#`9N<^l&ZNWO8U-r*cc~6B$dUI6Za6Ouu~*3hUQ@R`wFxOoFU('
            'Bwjpv^YpbgI1IG2gmyYWx&-J}N?@4lY(ln2G$Bn&x+^78I($;z=OWFsi^kZ3IY8rdju+RGb%&Cbrl|y2q2)r_-_VLhc|Gx(6G{Rq'
            'jVz9l#4(Wcq<KU_WFraQD5IKF>(bYD6J;-9(@x83a${7EToA+MW4AuND-@^kU+{tIQg+`A4Y6u0H7R557G!G#Z-Hrs(Zr5D$7Hl!'
            'y_A}$2Ftm}KK%DaB>dM5Pck*~JNjT;cjUw}nZ(%rE@?0pb0at!g7lDxBk*x+%&jtuVT-ZOK1rMK4qG#HQ#{oTo$2}fs4y#fv`TTi'
            'U=U2JzcPT2iU0GLkDtDH`s35n=i>XPe|+)$Pp@9PR}S<zcBZByU<RB7CRxn0BI4mjn}Sow!FC4eI24^mY0@bI8sfDleP75n`kJ|D'
            'VQXuB)9^JWj30<s*qJ<=Vk{3xnRG^NfSN4xvQAm!-2(2Ug#Qf@NuG}Vtzvk59$Ct{)bpvcy6w8<La@JnY#5N8FZgd15Y$Y5+W;91'
            'yG)b+{Q`=*<AOd{EYRnR|M|F>A4f}ou9S}6V5d8~p*Vo5JG2Up`=_H~fV%FhkMex~E#B_k4(?q}?YD7J|NHSn{td3hALYmVKhh6*'
            'nxgxW<)@=vHDBfj`C7X&X*a4&x-Ri02Dpgwu6sb!yhcYkx(ZO6j#-vN@gx+>coS+JG6-1c9!dsXF&GSV4c1U|4c0VO@k}aQLh+SS'
            'QwO&IMK&vjB+C+xJEi|W>JI^e'
        ),
    },
    'video_analysis': {
        'sha256': '7ec024a64fb138bcec02f4416b841daf62bb715198078295cfb575c36b342356',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<f0Nrbvf%&v6j-I|Msy@uGl}!|O`=?R?a8j|Id;l+lD9WEMN=X*VnmTz{+O{mZ>sK--9K0Ny{>-%AOM0I*_&K#s&+jhiAJN*'
            'XmmHa8^hu7P101+B1^<g8Yjgh%A;&wrImOr)<r$p7TYARMV!>hvQCR!WW{y56xBK^lUOYFgV||F4MWv1gl61on(p@bLPU8iX0vc#'
            'G<CB`!l<c>ZG^DGZB*4s$)8OJgI7hJEQ(?yW~XA4B)dwabtQ`1Tx^@HPA5x%l*Un=h((ew*W0Mvr1`b@`GpuQi|sBd@mH1HBxRHh'
            'eqE>4E-A&fh?7iI>BpoRi)xwVi5yH(!UM#u*da1eCSJb$5xywOq^zJp4BhPM-(WmgG_|PLiHMeU17Akjby24EdW$Hw*i>RNvHAeG'
            '&_@%e#blR6o53z!BFfw^!pn*!faBp+=`~_uP#uxMF|CRcK8X-|mfj>{os@~Fiov3&*CLlxs|0|`2x{^H@Dx?FO~ew0FVY<79KlFe'
            'VjHD-4gW*1akI<PCD3y`Nb@+^CGdZFo$V)anXYR5Vp;6cq=Kn=2h>DTHozx2QdmISB8p*pO0jDeSz4`=3Wl|>*U<1=BukNJdr>C%'
            'Q6vXSoHAIY00k*<3p9jQ{BBn!H)(P!mRY((?!$qUN!^q=Fj|^Vw#l|A_X6fW5>;I`FcW1m9Xx|U<AjYg9RdNo2~DeAkyj~lHqejy'
            '!XW{9Lnd!E{jCCqt_u;T)n+;v4u^wPS!_iZt{Ma$h9ceWin0b|0Xp$)H5kZeWupG7n#HaxmPu8qCw00_7}hST*IBwyfPR9X`~vwH'
            'KT~hi?=dz0SmYY~I;s$a`iV$uB=Tmv+l#0Y`ED?H`|jzxpWlYRJbn4|#al5KcLM?c4U-SB4pJO(lnuvX7{mN!#SR~YyDVBJ>mrMj'
            '5?}EYgk|!NCWZAq95aMy2jjcJ-(m>7)T~zNGR5XN8TfD&!IaW#_%LpW@k5;YJ%G3e%zpmqr`K=ZT|5upzW({mvy1ST7tb$VhZq0('
            '?&8(k7q4F#*iN^*uMyC6Tio!!8~*n;S#0T#=qBYCU*9nFA6`F$VZXWftAsQfR&`X?_|Z0v<1E3yfVuIXWmAFp83u#FAWl}uB}I0V'
            'w8E>5^6O;u2t-BEl*=THo04T!l`O}=NYEYzb`4MFuV9XtKt2B<C$|8$sca$`hmeqLA3q3*zXv?#v#|pkAD(?Z3&iAmQNzkiE?2NR'
            'Yd{$PxjJXkqO7Ok!g^dc+bEx`N|?qNmT6Yx*ArTuf=7qT0_1Ixr?@r&ev!v02oTW4hqR)#q?Q~>9$bVdwQ=F5rED@;L=`OGsJw1?'
            '$%%Kcte*jEfczz3&tOdzutuRXEvG6#O1yt>Qv3aTvCA5eL22HA%wRY)Spp<el-)-AYQUt5wmaw&@HYfD&nEmMh8b)+;&i4U4Y6k!'
            '2y~ikV7bD}i`PF)6~*Y^bY;yv;3-BtmBV3ht|yKr2>6ePKC<Tw<>V~TpQ@R&zc%L+8{m}cCV&KY@ST|9OAX??xtN_>+*3wrm55)W'
            'tVu4)vM5JG2bu4oi2y$6_WC^n&$dyP4%@C-40HEm-LaQ6NJ$`%wIo~QI*6@?i!@8~ZL&{t3(&c^Gk^Z6yqEaY<=&vUy+}1RJ&V^I'
            '%|Dt17%dJTm<8aFKs*xP%%(GHoP1a&yIPFi?RSj+*ceKHQW(BCFxVk4unM-8Dn;_6)JZ?82UWCBE8tZ{enI;LfK-HbM{7@fx4Y(J'
            'lk=7<ia5GX1xW2&Y@&T0ZG?pjPeQscN=fB(<s;KW9oplGqfNSrU^r0;{mo8dSmBFi-)w9KGbRBD?oQp<gGb=yy!l*6x!+xpdImFJ'
            'E4fK@o4Lq{iN-bn4dv`0RNQd7Ap0RoP*u`4vUuGiVVr%$V%E=KSpTFJA3=WxF`j|21<jvl^+9%ZCD!Nmk|)<dofUG?Hu`ADBt;71'
            '85T_2=)(wxKR!U@fB{H>Ru)YjkJ@GkCK!va;D7w8<-)I&#t;(jVY&)o-&iKF@5Lifdai-xE82f?W8l#)*U54Ns`6Z1C!jsnrT!3A'
            'tqvGaPq3}#ZrQx0oP029QEBo}YMvp#j7Q6DJQf<EcIO9C5CMcVi-l|=Zr7-YuZx_m4BRf4yc?f@F;NttHp1vZP63<kfajv?soD_{'
            '1ah<1e5XIb=Q;OY)iKz5bMxhkpDyTmQkLE4b@HK}zbl)>BjFiG{C>JCvTPLKcHfd%{daIytS?c!o>mzsDlqA%CO;w43P~6n^$B+6'
            'w#msz8g=)>b|BBd{>;GSi+|z=dK#8f{uu4BvC~Ol?kax)Md^b6#NAJKMIgXP1~#{G=*R~8@-|BA5e^Y2;H$uu1O~?M_@j#tJJj?&'
            'pf)Lz+eV~^97uadp_WUKcA1g+BaG8huTXx1Hqsn4DO^2hAa&GL<jgeRfep^P)-9OcfV*Hj1$j3TXjJBC@xppX0~JiqCYK89Q+V_N'
            'ZC$WR-@otl4Z7tmsGh)U2~O$`zD(k&cn#f_w<xHv|2(;cU5LPeRy;|zyLwL_gBb?uCL;#hs{<cK3*dogt0YU7HNt&oLo3myh7ZtV'
            'fYXe+^-_$s(ilXX)Yt-c94KuYRpuQGpIj4$Tmy7)SYnZY;mcYE+BO}XwnK<>gI^@M<}@Rt&Zm}Mib|hqF$E(E@f`&|k^cj{c~#Uu'
            'pmbFl;D4A3$)d>`7-|%!paCkF_d}~Y-4zJzdKADvA^LZYQH0&zimIm@xy02g<|DPQq@|ZG@LUUTvMBWDNEG<Tq{)%=*nqiGl#NZ7'
            '7BHgBdyVpqRe;)MyK0dS(EUMw0DGD^0lyJaCT;P%b_(?yWIK9A#{Y+DR@~x7gb*LJ?L9#3c>|k+zPU1q*AB=~qV#eD{R&+GzZIl0'
            ';-k@ncm%<N#J8AdH8crL%5$99>T^3QQ}G;?)D0|DSg_5<D6dmcd~vcA()U9RUQU{1GE_j?;f}$I-p@hZS;P^M>s*W+s*!96bNG~A'
            'plDDt0)rfUA>N`OOtuB?C6n8tjKw->N|r33^`Sh2J^wmIe`9WVU6sQTchKqJ;>XwTfN<fjuird3JT%q1SZ<K~^ap&7e=m}{jP~tc'
            '>}tY2ME%SDp%^UxwfsZ-mu@5SKCRO0HF<5oTm#bJUv=5==j3$6KcYoJdm)TWHP4D0a`CKze99XICJ4EskSf$2x(ev8MOGAyN*P@v'
            '5a?xHf<~dhrRyl?-m8MfS!B(Y{{?-Jo<}!Pg@0V9A2V`B0qkW$AEOGZAl1n_OE-kbCZe8_-(cfklh3S<aIomVqKy-DpfXz4A(^1S'
            '3?aFVN2}F#mt2R76fEMZhMlvFZVfR!W=0HQH<Z7rS^N8tS4n7xz`2nABGY2l50xTCekw=Dh3({7Q&-|r!y30>zqAZ+fm?R5az+3m'
            'q%K4xH`WquiI1?r|294Q=Ih5#pFSfWh1>@vjnJQb`pxEVAH^1QSMlr9U)c7jfdS}q0)|o#KF_OVnUYfVm>uzYI}~qF3RD7AV_Z)p'
            '0Z6rgW=)PdPCSsBA!>x6SMX#A$zNI-wx>XC4fa!(J@(|NFYTv+2j+Aam3W}C-Jrnc9~G(0V;~Bw>#*1u8ng}Y6B-Y+vk~-HYa?m-'
            '5LkCY3w^Q%9K%JF=e)2cS#h1AZ<!b!sX1)fZ#Y?@0YR(g%j+cH)%52CR}b*@$YzSbY*<zD8_ib%wyfn-i^t4%+h~Vf%+7DlXVjN|'
            '5S`EdyM^1$ik3AfR>_<uf0W)nJ_{b5oy%Wq`en6Qo|9iXOBWx2RjGp=tP++SsDQ4bs^-V(hj6u}zxU0}*Jrbx1Bm276fNoN=%z^H'
            'knLVjq?u0JXp@8=ld|A}6gMp~u^Z{{HjJ6l<-f79#g+*j9?bR1rk!8^UC{=NYn=QMB7D_k*-!GdR0^a4=r=(n1C3=zS}AhuA@L`V'
            'KgnE;pbRN3C|#ri<ZuOTn@s6J1do^J<>af`Y<7O76(U{(zdsu*hIPq{T$Fd%%xcQJ-03%fD9<!a_$mD@nP9ivhSljebYy|OIHJLv'
            '#6vTn2miZG0XpwcST55|RHhHk0BvbX`qH5<Zf1-Y?^ARxOZp#eY^c#yApDW)adML`+sDgYGwj>P@Z7hlVU_H6Z3Ea*{yZE@?J7+3'
            'RdEiQev!euA0lArK4A7fr}wvadKi^xN5DJ9icYX6w|v+PTMG4AiO~v{Oo)yRn8)Or&OUe!Zt3tgiI!{i0vFufi56}rV{wwe{xAPT'
            '82E2=a~%R_0*;Fl9>7UngwSiUT!;KroS>>#ot%qH3tkJ3K7fDl`v{%B_5S4k>fW{tX)oo0g>qo(kWwD1|A=$P%Z!iL=hmB1JGW{w'
            '=lsvuny|U~gQ=rv6{2x67C3k4<3o7UmYqhHpljy@Rw4Qio9H@`M{SEF+7cr3C!i`tG1@f%nRh=81z*6v2ud@&tLAq@&@iDD=<?<v'
            '!nPWoizl<${a7FwLtG<3<?XI=z~RAHN=*t}nlSRBsfZco%zu2~pQVUuW;70dFX^nvh_dfWYeSmfa+}WW`uDlfOaWrTo<=*27#PXl'
            '(su`g#3iV0v*db;&p<|i76t~R#KXjaToKN9)-b}dxa|(b1Y!6Kac1oARU5K{@aPUZqm_ckXqQ)xqXMwOms55yOK+o)?WcG$>GeR$'
            'iLcYVNvszdBSo;gPP?IAtiWOHd1eoPiwL`=yDjC?_O*joeXa04G@;LAA4GP~r-rETbz;1xBhKjz`uy4fbl>5g0|e<M22NWaEqdsH'
            'P5fjSBDcZUF5@|08B*5++IZ_3_0X2jp7!<!J`lvFa;Au8?fZM<q^{lQTnCo51u+ES&|8VjA#I=|byk7k#>QJZ?8Z5UfAn&OpWWaE'
            '-6r%gPlv^r9zES!AjZUBem=V&92~V&NjNMkq!_JgWZKaxD6CaGU@=2$c1veqCS7_WE3V7~mP-7ze~XZx6D$#3M;Or{+HR&sHz%@#'
            '6CBS;6h}MM_fL2)R(eE4J)_k;G7N0V>;WUe5g*mm+P`Up+?m?E3AG=={JqAl6_0D!;_^0{>L5AQ{Het7xqLCk&<z}bdcX>SY8Ngu'
            'l)-nJj_2<8X!A&z@X`f;KLw5$DQKG4E$|&dd^o<Kiq;gF7bhEl(17}?p#7=57o&yr-kl4H@>nSR0%cXOIWjk>H)@RmQbFX-g_I1p'
            'Ol<hxr2WYA$ux@NP+>DN2RZUVoO<}!{EV1p?z91VhVAQ7pyq2OLK)vfCr4-&1P9Kb<r3pns~RwH2_@*9J9!SaI-ryQP$1b4hydlf'
            'x1FY!Bi*0c6!ZfW`BK&3lVRGmbAvcnQXVJ`RO*mPw5cpBr7m_MFa(~l%*E+!jK|Gk3;<9)GMr@icKYOiw`~o8fV-~Ioy*XkoSu<r'
            'iPL4RrN~Q)sWhw%P`|1O&lV}7iM(C`IntR79P@N{Lb@m@{`qd2$5B~Ed$0z|ZRZh<kPQU#A+uG0AA(jpoldXr<v|B|*%y1kr%<qF'
            'RIF%>K~#Am>f1sPG>W!FV+aWBQb)@qDMP@FFN-STi3EHqTpRM?6yr%p9fiIs=VUQU-FRCWEmlcHL7?z-w4VdKmdf+?&){;#iXaim'
            'Y>pZ~DCPyJ@*^Nr#NmI?ZUis`?!GGf0P{GI3b=NoG;_4nH2#2WM}V)9rUdo!{CskHWgPKhdkO`e8>_0igw|Kq*$i06qi9i$+6EH='
            '`up~BsOEBb6<}<aI}MBse2LhY=sK#e`AREiEpWAi>;5@ZVpx)8WGXlCgwY?1Z-RSk5J+1K1q_zqWoF^BC^2BVgB4NHI3U4Mop-5V'
            'N(vqUqDcR?2ygLk;$iq@_M&|97*vCYGdH-QC}Qysm@AytiWZjq0M-eawL}9(AuDcydD82(o?HHOieW3rU-m=jE6`gVHJcy^@trtL'
            'CZ}f|?G8{tM*}geL=t7?6KnkP4THkCrBMqx(5DYOJagc{=x~8k8I@^Ljk-q!Qu$E74<0ax;wC9G;3P10Lv5_FHidHU>Vf7guv>Iw'
            'Q`Wu4`ea+%=*J-iWR@EtsEJ8l+2<WFQ{PNy4<RPicG$LgXh!SV^s6TiG+)6|UPQ}HAAo;*(i6%%rtPBDIxlG6Kux5)g&5-)V&E4Q'
            'AVHD(psH`T9L=ATeNMVwTFc;ww=@(!JdxAICxj9fC|PN~%o^AO#40VTnj;w4Jl;loKAPD=Qvyd(|2VeWsH8nB5_uApm`Xv35pps?'
            'SaWC(<txAt5E23*07`-)jH&n)JE+NTM&1tvumuI<N8h2Qs{p#<w@sDAtWZ+|1Nvgf<3I<XSxkVEs-7rFVv(#Pbk~$jq-9grDl`gJ'
            'KKPplS&C&=RFrSBE^2kM73HGPK_k~i5o5N2kjXHXkIf8vRN-eV3OYP31%f)S4Rzp~YKNjk+LXotNndlMH8-{)nsAhTVL=%fd`~Ux'
            'NCRgWa<N|;);#P@6ZN*Ud<>n@aaH+5h!tYU&&XUbdJ9ZXnc+5<UBeI^{9*KMxSul!<z0XweECNYLG{%)0J(}oGXV0^oy$koM9KY7'
            'wQO17(qxOa6%yx4&Uh<)DSDMcE$(QdTWUQTb=jchCbhUMb|Mn#Wv6xXwl{W&-mB9l=|r`#xzz1lV8&-aUj62~xlKK#sb#Pe;@tdm'
            '(7&IX6u$BzzL$je3@BhXZ+D>N<yB$59c1^C1$?6jI7r$vJc6ksk9EYU#aa^&_TMQrre_Z|6v`+|uk$3<eA)pvaKaF3z$UrI@eK91'
            '7FX<to)~Gj8_j_3U>I-T&Km>=P;na!2W|9uXDTMv8v6`l@zU%<ZVhUb4hDKT4e@9y$)#_}IuqqDk0X<dMGuj_5YLkpj`s%RH*bLT'
            'BGUf|-^VO$7&VfVr~;7sz}hJ7x~$hoXsnS;#S8ehscM;tNLoX!RT8N-;2y1X3#E?m9^By=poaBq>}<Bdz(Yb{GvmqF+F-fC7K{nX'
            'R-79i*Ws=JI~u*ALmvM`l7AAXC1;FA*@Or*xj8kDk8PYRgUprwrn7r!BIa*J_*-q{K42{&tF86saAUVy<GI_=T72Y=HXj&Yn*0af'
            'nLqif`)-)OGsSI?zcr2cSWTR?jfe$9BX4xnh86jZ8f9pbg?ZS9>9Z4hO&cc{P(QXjRsHq{FCp9B_)2C!lo$wM`?O{UETMnOjQVT?'
            '>S5FAAi`1=3yG*^yR!rB(mW)?A%xN5skq+(_|=o?tUWBS?X+hYh<#kLx6UmWqZIS8DFKtS8Tr_sz)T1XLz&#1ogmY$!Qe2XIgPw@'
            'brfOdx=VjC<+}${@vPV`(j0UHDVJlM(t0l$cmk$S`-nVru<VjjEjTOKRDahl6x?N2I35fFK;fPxA82GTYz`%UZ3Y6~CKw6|2Ip*Y'
            'I-4<)0^<;>EX7HeY%Ie9ugfHfQ`k2+#vaVR9T1FNez+=RAJ8#I?i<Ru8k>{=5~@mb?ViDy1rbv0XawWijO`r-nc}e&dSf$ey!)01'
            'hLI{!g*$OeE)J9c#Xra09X^5?B3RE0n4wp%--&CaKGs68^X9PtL6YdVp?phWrPU~~tx)2mP>#BFT`C!&jJ(z{%9=fHeXe){)u$CL'
            '@|^BV{AYG*iWcqMLXVZz5BUL&P$VxB=xe(KpvpMWE`IWIf@*HHI=A(eWT{kT+gJuXThs%OSE<!|e1@E5Mam;?Ge*I)S-9~A@Z7L5'
            '#rMXTaH-I1!;Ti;sR3y~ZKQ)I0^^boyRS^FydzPl@aOuJ*9xw{@s`PS1r$SnbUD0yI{6>b<m3POUz7j)zfZ2d9CALFjz_m6Gw?LG'
            'OEG+PX+;`b*^&5%@$P)9B0cbooos48MjvCahd0)PJ~I378s9dR`G{g0RU1qb<%0#V|5^u(_;2d5o~*#-Z9dFEUwI4es*@tF7#CKZ'
            'V&jQ04uempMv}O-qCYjh!tfp)8DG&lin{R?n(fi?j_2|Ad*4LDm%hiIm<Z4=8#+@aIfAq6_AeC-xfNMoipLgl-G-#nwC$>9YsiZ4'
            '&?nf=DX|+q@pv#E=!fT=_t2r#BJd1qhp7aZJ(7%CJ{gODp9O&<IPow^dk5V4;{DhehgukRW{0_CG=-4g4C0E@{s|7-z1<whHi@`7'
            'uX;w`kODoJOT{46iUGhROsb@o*!c0o^k)?B)<mDtmW6A?ikum!;8EUUH_6_XO7`rGU9K2+_{%g-9MX^bXf^f?)(r7A@*%*61SHS('
            'k<{hwCH!WQev1=lt^6zYufZ4i${y#vp<z|FZR)Tj=;5<*cR`S@>{(#5>FJXLWRM^9GY-6-js5cvlla6m?C3l*kCP8$eM*tTl5Do5'
            '^o%T87+w}vr_xk4Wm-#gqVOg4bA`+Ws_0mpJxJPC!e0*E9$^!nchKmxqQ~?pBSuS?CyZzPQ3poL4H7f?eSjFZksg-$11n_(nr4X~'
            '8dOTr#;IhvFXx;ATNtW}Y-~BmCvs`7Z0EoJ)I1O{9B*rMEmz7P<Me}vmAwGXuRMLYXQtNhSdOU9$9T;8?*_S~pk=dfgs03w=L~kl'
            'htH;d_N6?KHmJhYA_g1^qCE+AxIkOqj1o1;)o!t}l^STCd`pHwOM)yNPUkVEw35pSV+>Gj8+Z<BB~xfDzy?_tRgwp=-L-y9>tkD+'
            'qLau)HL|Iu;RUUYpk6WjpksJ{uRW)SOz)th_%T2vpFw0J>(bEsT9Vjt2acP_at<jEtMY9DF?$cryTQSV4h|O^&)ap?-+H<$b|b&`'
            'MEY&l;nW4zv2ezH&YXGA^DfM;LucN7)NST-=gfT%?WXqB8ze-7oFpe|0R|JGHG10DsAInyvR9tUnqQE*`w*;FD)}L)u;e~CuG7E('
            '<P>xIQgmFK4*>1r)(cWPMf(7{D}q3}4}0f|+#b}#6BW$b$o#HtaN=%yCs&szrp|J5bq@-J2NZGegP<%$rymw^^~1m{^{5w`)RG<q'
            '=V(&^Z+Kuqm_-hy5!i}hHyX=YB@YE-$xY7`uk0o7@i=wzm(7@K1B&fGS1_<v$e~CYD`Sh@?%^s;vRFrPOFka<q%B93=p_E_Y__u;'
            'Y2<8dgE!bWX<{X|4Hkry@gS7WWq1ZD&JIf}a+cVMYKg->5hD!vR*xvd@Fb8v$M^5sF}#1T9h$f66hn02RhH%nd}S3cNh?w09F+qz'
            '5j9yY%Qnif%VMfr6(v!{`?!jTX!7M6eS1K%VzmMrtCHn2qZmsC%(#LdA7&R7r8Qwd5Y<P;^EHYt!V-U~ZY+{-Dt>6PY$B`f=~rtk'
            'RLPE6jGf66yA_s&!NQ@tsKQi3DgYYY8Z}>p6ktPDKa~&W7FCfob<&nzd4xg?2xATiLxEI*!A>QVmguU0-AGe!ta}ZP4Kz3T@UH#g'
            'M48JDR56@DRv%$042m9y+Mzl>$66FEoO@$e7`v~OZ2$cdeKpgoe0Aed=bK|w)a#;-GS4<`pgkoY&uBL0kH^5OX+0lqfK!HeNLN*u'
            '=eHWp9^cUi_pl1mk6E%!BqdQJj7c!)RcsiEMz6AD!TdBZ3oo@xRz)vJGCT&Fx2?lF{`c|(b&r#)^VwN^PgP9YpcwvOXu_mz&+;zn'
            'E{xbPZYcQx4IKGIGfC?&bW@#wk921z0l(9EFz#eTJL&rjLFMYZL+pHLCgM-d&Ot8A?C1lz_8{-dh?DIdXijE*U)sEdnuGt9RRF6u'
            '<K4})?#Z^EcPbS1vPK7|_An!JZ59^@Pvey;Vy5!5PboiJxBNmp#a+9>9fg(}9&xR(0M!~qr9Mug*hlz;_cK^RT4r-9b7TO7T_Y3H'
            'feDL~7rrb!)mhlQG!L)sJN)S0x4;L3FV3x9rowVqb%QaQQy7^l)yn$_f~S2&oEd59mD+J&V?uDS$1f79g$r_`Xs*{(1-GDr*`O-i'
            'uCY+9lqFP+7VDF9SgTZZ`D~N?2?$U@!Pr{eE!Qi9O*dXO8PXgO+@b{9SD=lc5t~G)en6|7ureERNK2gp;a6$4oeqBZ`Q^*-?X#z^'
            'WVO}s`Oj~jl5_X%#WOs@|1*Y;0hqT)pKU>V`!((2*a27lV^!c5qjQ8-$hbav6Qfr~QZvTWN7yqE9@pH}<iW2O_^G5%k>i}m5Fexo'
            'k%<qic}L#l#cl2xz>hz_eD~s~moF~f0QzUf`yZeFr%~nk)Oz*$$BS3*-iAM2ypf;afjpx5EBV1BnZ<QhEMTyE!3E=~?mgQM5;ZOY'
            '2V2B7>qxq)7S18UyKU3=Ipn(4g<xyIV)aa`sd$@!N)jTdMAijW`^|fE)PH^X`Y%skh8M3c-u(6N;jb4j{`$9f6p`??QyGX#3LIG#'
            '$f*MKPzj$4?t)14>)&3#ypTG{iOfYpP0*;g^V72*=`0SV0<0;iyZ(gB>y+Z4#mZfnd6L8xE+;&lc9@tK4R^%rwT&c~vqEdaT9lNx'
            'MXoE*)0?uweu)C!x<2w;+pO)Q>iXQu=yFvfnw3-`FGXOifW}F*wbjPsYx#V3IUe7({`*xhaZB#}f3INjN0*`beafH6f^QX;H2#$8'
            'C&>1p>776XsjQxvU`RP=ZX)5Vn;7`4h|qic=Et{6Uf~w;UN?U)qGefBcm;!6TE9O1Wz2RQ#w=&Wa-&x2b1)+pqy@tYi&aDkSV$^0'
            '4y^Gaig}}|Z;J`8Fev~~P;5{hDAkqGrq~QvmayH6b(6;>E_b|dBDszhdn#+k<^Nbl;L>K15XrTkOgPVR0^^!+A}0)lZYoHyt_q_='
            '!@#^m{(}u$c{2p)cCw--^jXFkGEt$bTnbLvb1Xm3O1zD3k~XTB<6jkBg9nRV>reYjTvH%`75CRV{%JPT-q$3v5Wy+c*X^W74$H-m'
            'Dmk-=GK&SHJfMt3DZQ?fI#*B4ppG;q@zs8X6G2a{8n1O?eMG}9?Lx44zDv+xS)DytiBnm^z*v(7C}FUxjAcc|T--6BN8-=-_|h%_'
            'O2`~j_Wn9q;u+#cQIt9MwgvH?el$zl6CZ_~2`Ue}PO<JxJ)%w?b@;(Jt-xZ*&6W;lXHLOll;19Lw=Ml}RdSyr6M;@}3JO4I*sV0u'
            'I^?wfomP*Gl)~htMYCEZB;{3JBD{csoqiKc!S;a%ct}sf`|1o2sJ=S;w{QMzR1eo>s3jwPFHXCN$ZZm9r85FkD(U(7v2KSar|R$S'
            'se#5?{&jiI5U+qn)f)8R(d2a8!s4KU2)4=jX@^=C23=FjPd8ed%1NptlzCns%h^X(AZNYZrCPeGwT^tK{!z9O5cIo&ZC+6*2m%L?'
            '?(oaGH3%z<wcUnQI;)X@Dg%zq{8MA!=6qM<bA@?t3(fC3prc&@#%E@DixV2eIn0w&JPkG40uhzN&5ZzQvu=TGa?c#eJ@cG)!?AcM'
            '9ks36^&<cQ!qToOcT7?i;i+~Q%B_K78^@}a2`hw6&gO~QD%yx{)>jZI4jbdy2*&nPBza)E7b~-r5P~s&q7#h1af2Y`+-o;cC*aJD'
            '+_K`%>=-SpG%=P#8zHo5l<!AD(BGw1qe1aB#;j4aTA;NZ@>GY~V3HP3<xZggW}V~mcBK*JX^55G4%xxS0VcPE3ZehHFO>VI6ci7>'
            'oQln8=~lQ+@%An4TNrEGnUZEZvBy#s`vI)ciV{@NtWhAV9qDa&@(qfT)a<ZNG?%o+qVr){WVY(WUhyJGWvpUWBa?CUQ__m0!<ml^'
            'bj~cmdUL@gl!dyEf(xm$@efMId!A+_7U!|NAIl^h?gx<F;sO1JZgVLyFPH2$TkZOUTU%7EpU=GlrH;0$VsniJ-tbBZx&`I%^(Q7n'
            'I9h@6Ca^ws?lF-&N0()T&I&LJIoXgN8r`DEt`ju~ulD&*$}X^lr66vlj%k-sA(CzPnIX`=7qfvcy(Kmx9_t;K+dvhg4ZxA2s>Q)5'
            '1fCTXJcEQbXJl+nmCGxjlQ;yXQc|a86w)rrJ(sjR<_-t8yzrEmqgXLCw>Vs$S_S?qTo85s{1J6~@cI%L?_kgsUx(_;QAfjbiQkvv'
            'bbL$=I^V@8-F99bz?hge#-uy0gMdj5VAaKcLE6xjMwnIC<SAa47%H(g>@7v7vcs5qGQXWN737k4mm|e+83EUw#0mGhA+`n79XQVL'
            't?UHzA{&^Whe&5%hXBnbDP{4O2T6}Q`E2^+ysOUQLiZ_PW22N(NAZ>|4Jpvoyxo{|i{B({Bg&lOPD~cyKJGgs|9;+2*sjJe2|Sk*'
            'Z3!BQp__t2^J}Bqn-yma;HYWnfnp5XKNqwrU7>J)CBvF22+j<scsR`Pp#VdTQGbk~t{pPRYd}YXhhyJhP2h#7No>Y&l9xz6wZHWR'
            '%5@?DxhaL_5*hr#e+t+0!NKyK4C3Jk8WkrwpWCXWhY^h#I#|o76Tzy=o@tm)R>uP(zBlWm+npzw<A9y#M7rvdR1ZKjSiB3Z<u)oH'
            '<&MN7_3dzdrzoToF|G<>wV?oV{-kSKTcO>9o00v~4O+4+oi-u2&ysi(<s0{ise8Wk;?>(1&o8>?OEO@nlb!q?Mb8BGuQq*v4=Qz7'
            'S(^?@XS--gSo@hgE93ivRa{Oa{{4)L;zZ&fO`7O4L9RcfUaG?zG*nKi?)f1}TAbgu3%W6w@=xUz&U7Bi>F}}?&~4FVacg6&q8oHb'
            'Q&bdL9}@hURw|duTi{7;Rdcv^j%R+ddRLjMOPFYe*^r`(lE8+-sxU-+JuNT@37xZ0`4CATQK=(Mp1c=tTjzpdwei6**>01GoD0)2'
            'ft2Ip40eyhM_l4>NeOp!%`@G9g<f}psB(r286}m|P1tmXtdE>(GzzEU*OV@aR-QV?Kx_?JIX+o6Nko}c=9$g-;8dMt(5LU^nKfzs'
            'u#VR9%o<3K$fP;OZ6{8!mGV5<TT&aJ<1Da2pRdioFRDO@Zt!g9T^pOv>z>>_h!s^!Bdqu@k`xkYkoWRotLmt-LS(Unm4uoj0i#q1'
            '&;s}exxKg(tDp;(cbgmEc|LQ<G}ii7E<UC^M<$Imwr2dlsiP^clSCPeOsBY<8B*ez_ip5pr-7ToBE$LB9PnFCPU(+pF?&VQ$}#9d'
            'c5(!ind3OFAHv_xVr*a9C>=4djs`aZ`v6cpe83;*lLeO~9e7_p07^sZABO5fa>^&njjSCOCnX(oNNv&OxE>wGrkJBRN_$J*ig%>8'
            'Dlz$VrL!AM-|C*OK<MPOI3M<Er3V4o%BdRnKc$}PD+M1^P}P^DqM~nvl#`%WK}BS_w1F>jQbF^DO=?zCc^LSIsle)y%ljgmoC8}V'
            '@l<fx!yzsuAJcO*_M?aDHGZzoSzdq<daf(UB2XVDH(yO)T#I6OBp~1ACi>-p>r9R*l-XOu{OIy9f3Tw+vx%g?g>m^W-i_ux9v&yy'
            'Ux(nzc>iuxlNB@Z1L(6x5PxNaNL;*n_u|b3hx##ZjHy-w7B!~mbERG*e=^r8&U`PD57Cw?r4%yoFTHMxrc$Y>qqDDP6~-vYV0?1;'
            'Dv!DmPeJ-%N;e(KvRA0e!kBhojcA;eRxMV^t-7;b>z^`k&(wQ&1=1D3Mp&rR4IMNzQ5g#_b?XHKB!gf<wb$UQ4<!Dit`n_5D#WNm'
            'o~zqlK%eJ(X)7a;AC+#UHc}ujSOF;(+Jq63@~qfO)gLXC+KUl0mAr<SuhW(19a80%QD9Qx9!VXVOh4o7V@%V+Y8sV{p;(f!JbFhN'
            'sAPI&9pl`>-nPa&k~1KGDRr}AGv&CpGn6;n;Nm}DynXlL)n5_%M9%px-EnFLsP2?MB7^qO>qDRv#W<b=lkMB<V`R#ntZmQfcxI6x'
            'RT&6iOD?L>VAYr-^ML)-8j=`YS*tikU<+w9IOm@FYyc4-Yg<BHA-j$8bcMDX7;y6TBa0_tzvTd64*t|upIO)S!%kLzgAs3}xj@-r'
            'TwMIUArIa=Rd=*`Xy!8gRuws|IEq%JOdC9v@+?xOItA&%@_Mf`9)Xr9T8GA-Gkh<jiR3`3z7Uz>7*?~y$OFfS`5gk+jNMCx4jPX-'
            '-y9n)_%C|2mgrC@ys{ZDN{vJ7##-aphE)^mXl>EN*9p=a-y1xGLqN+(W9f|*=987PvA^#};y?g$pz7X!0gLZ*|C8d)%xtMJI({UX'
            'aLHGLzH2l+Db^5)nj2@558chwh6k&FRFlUZTP#1ljc)Ws?&JrU8)edsKoLqP?#7ScV|TN*Vn$BULG8K?f!prYwm{*Uu$OF2Thh8~'
            ';$YZwR~>YQZzu<DUPe3Q{t!(EnsrnA$P5r28*5@LUcaS3C?F5MKDTl$WVFex^v9s-YK0hM`~Zlcq`aa1RBx%@;%@$1Q`H7Elu8)J'
            'k%;RimC_V_Z8=`;4l<ON2cCiH*a8eD0NDvT24-fc9h$+BOG(N53Z-;av{@U@bYa37MvuV&gJ~Gz?ZaVsPxojqjjUA3>R6C&v>H-E'
            'U$ltwES2Iy?4z9wrNzwp>^ebJ1~?xCdvu-R^~p<E3!^);`+Joow@S&UXHuAg88GggLp|wUSiE?B2~4<LpIc$6KmLUsc7PpvA3>41'
            '%~?UPvsq$IO_>n0gaxoSoUt@{)Bf9197dWpK!AKR+NKp|K-$9F;4#Ta&h-o+M+O7?0J=Bs#$|VJ+|As1HXa->0p^}(W9z_UZpmwZ'
            'E+5^k!8isg?T2RHXW|%%KxVspyqIS%WcWj}OtV-AJN)yK`pk;zp#9mAq!N#z^Wo7U{Y7@j>tWQ#iEbiT1{`2XR|mQusB5xpDcX5$'
            '-lAoGDsksIs=DYnX_P9b%5(-uTKIec$J{1pZ-^2fCHeZS-qr9gH1#Sw%;t?0`$}l*^rYNN0HeD$n&;BEEb|J(M825?_fwEUn{*Qe'
            'MzAndNl=?>%O98zDu3Q7h;-KEO$N2M=ud#6_iVOxYAXGWlO=}{^h==!FBk9br+Tbo1LFrQbW?P=wR+@3OC`8EuuH32hVA{SW_`?1'
            '-VEBgVzyiB#CzxLyzAG|SC!hH8c!(}^9oa+=f=8vR*Hdx<HQhA;|{`Z@I;SDK+v6U)0J01^j=!oFHszFb8wix4=^|uk|TT$Lba~@'
            'vwJv3nrK%TZjJUGX$C(lH`(0wnt8xDHv(Y12};AzzEP~Y#0#CS$cW<3$?UsyJjuj}l5Y>+I~H^!)K&Oi3*}Pk>sMpzAsw3!4p#f|'
            'L@2roGdOCc3n$cD7bujmf9eFwcc4PbOQ*7SH$k9X{0C}kdy2Mp2jG=UWNpq5C6aclOtn7o&M~Fy2G?kD=j0K;^9TK?01kyF|L&;}'
            'ZijPt2Ig<&#&P(Ca;NRRa}FWZmnoHmibq!M25W<wHak>GG1`k}%&gkUO9jo#R(UnwE2xJ!(W)i5j><~5wXPB#^Ch-mJ;+GH`;NN('
            'IVK$|o3C?BF4wl4)R6dgf&v39%Hmc;w|q^KnR#84PfU51j-uMkORL$~ONN6tSQ!L4Y|Y)fk*20i$NIOctY`%+*(Q`FNn^S^kzJlp'
            'ilkmnt{JDx{j$|YrC7YGh<BIV`?og<>_Ow}7$eLw#vE%&>e1>vjs_kaDt_=u#*kwWd39f&5;*~_Q6wWATf^f}%Bxf6wd>{sp)1zu'
            'a*>v1MRYXDQklPpM!lnZqsr~WmE2r$8vHKy8~M9EjyCB*M>pG3Kr@Tv=?fOE%8+kGm6lQVN&YswjH5qXOcQ+En&Cb#q`unJL|UGA'
            'Rb1o8EtJz1!w{P+U8JG`r4#g~;|K6y#jdVVVAjkT9CuWQKH8CEKO4n!^dN)d3Vrba%u-)-U7l$mbEKpnYYxnqX0fSXgz9C><W?CV'
            'Q;w?1fwhLUV(_eob^Xa**Lb)$7HnHge(v6^F{cYPRBhJz^NSyz{`~Tt4CHugcwa2K&pXu}lvbf7<J>+!wafX<`936kX8UuvxUAy>'
            'vSKw&zY?+S+zC6?+)V03m$RMNO_;~HS<=@oL%Uz<*iNIC!~_pdQS#b%<8ZYzR|B+yRS%Am%Zlr1gGOI<>r%I~i2m*Fdv#{%85#!$'
            'A0o~@{~1^ujDfItsRwC|y<~8j*e6|WQ+3$od~U{G&`E@?p$<Jx!{cZ=<m1^8(S#Om4SG7+Y$JC;s}E0Ie~+Q1m_A_VBR+Oe+7Nnm'
            '=!#hux7tVw<n?OZ<fra=-K;Y&lGQuzPxsneZ?>0370W}aRdD&c^Ia9n^GC^4T36_Ph{p;BYsrG|r`FZKwwgj!aN!M8!&NK;PbV$D'
            '5`;P3y6bR3;k3LXO2qX>XhG$$)zH7x`9x+yLAlOVSgIlFp@`JTt>EvNl?NiB<;B$$!b(G<gK(+F4?S@Cbe**+T|!6(mZ(H5@dK*0'
            '9IeJxD=4toh_!WfBRjU(Tc*ek8Ky{as%+$fx;q>$c>k>xPfUfFP}v2f%qo^(?F?N>-v$p-2!$uKlB13BHq_`qys;N>U-+@F8(Ol!'
            'D$^E{$yYkEqK`pTaob0H#R>pal_IImr!h|y%R$zfjK(*Dubp9xu(FY3?+4#<V*=BjMX9hF&>j-3L=>%R%1r0B(+=MmRV@iy9mrv!'
            'ZEFb}myb+^2$aBpiR%s@QXq`1dJ0tUlJnl#Eqs1R+A(}D`rB$BqSI78f4HUPC+vY@eq&leK3t6eM%^v2P~G56$|}WQ!cO|F`1R?V'
            'S1(@u_1wJM3Ij!w?XKR7QC_t7XVon6hGzJq0zViGmKmNlS-~MXbnR6!DR!b*&}nd~gLCxlUKsc#aGEQMssYonE}G>!iK$?Lu3yiV'
            'M3rPK3es&W>kW;9xMDh;>K1IBato!=#JWn~BW0uxj{rqp0)`~NrMSeNuRX>JwspRixjEH3I(za>NJsVL48X4dHq$RouUY?JN7Wi^'
            'qbYwr3fhV~R7{7J>D~%DTu{ErH@J1CSg)RAoX^cFE~C@g+1C_-e)cu&^^4)qaZE5W%B4f_tb{K4ew}>azecK}f@!wMI8HfkQ@0H9'
            'qC!qNvWWnE@Rl+_hzbJCIa$wX-E5;=7b{MdDb{qy009=bGK>$nvB~Obo3?@FvM>Z3L$IcCv)xr*8;bBj_c+G?gi*Cj(>a}7jwyJY'
            'uF2pd5hIKlx460}tNCa+MjQPcQ_^Mh<Q#*4Svj6E7NcQPuO@#UT4Z#lTH+gc=*mW+G(UAN*=XP!h|J}(Le*7MR6<DK97{Y<jKb^1'
            'o&%QU)89IcD#KWlb~5c#I%;<~)}c~qsm=3qD%v4WeVjg{nj1Uew4O3zz_T)dlo?^7V2JlFNhsRxJ?tTMx{8+Sx_FGTP>*mABPTp1'
            'ToqDzK_$*`3{|Z~t}b;=K0srOsCpZsT47kfbC*SQ)pC(xvKZCh%p%+-Q8*Y$TO0|)v9&vp_&3PJAf;e8&MC_UonbIa<?V>Q_pw(X'
            'JGT)RPDOXl4X6mD44WLmA9Yq=8$o9RIan*Y6Vq5F6hEixRe`-^0lU}CPLoy2`mhGi;ZQ;LSXZ{G0#dTkp<q5f_6Xlx{PN=B*Nf-j'
            'U;YlHuS9|IIUm2(>1H4;wkbvq1Bo}VXu%>*mAlFEzUsw-3@-!sd<)19ta`WKbzSgX&wCgBw-+xjp1lLS)i|EM`VX-yKBhHZ(4Ce7'
            'y_gmHCskz(j$s9`1LH^v=%^5;@BllRAvjlbun&P;ez-KJcX)NL*~6lVtTXQPd|;4{n(rI({Lbp*-tfkL+p!b2$qimyh9728i5$nt'
            'Jq|{J5z-Y{GR?RTp6jK-$2xO_9QR)7hn#wOrEYeYb(TT{=q0r`0qK`i`>mIFSBlau&|~YigMuI9O`==e05x4)X!mVlCwoc_Ob;<D'
            '1uP|c3y$`Aijpz1y62TQbkd}BNpL;pOACdofv0r2P7{*192$kD?eguscbf4*`VE-QA&F-5t-7yXb%o;HDiBYq*%YiHQq(-f)h)Q5'
            'rl^9C@SjkZLvuPdARJK;aODzTZ98#M?NI=O(4)GZEfVHwRfkZvBHF3eon%zh$KJ+|Uf~X^6kClR|2#pKE!!<qFLDR<Kb80HvLF5s'
            'eH5pp=A|FgJbkRqKRi9y!V>`t0k33369aFGD9z>HQ;<&XRdin)m`dkn@qwXI7AQ0Zojk6dF|@A$2b+uhg1ni>-M7jjI~R(X<&K0$'
            'Npvr1-kyMoo5b9`Fi_A0=-HCcb7roo{?EH^&BN_YW8SKMVlB|l2Z0|#TWZAHUcGi(ckeoK0d&9UgVBeA`@S;*$_VWsPEXx$^<>*0'
            '%f|<A9kp8+uR3Q9rc6BVmfr73I&u?tQ`UyIfB;bTgTAk|3^QBkhkf2?&)!N7T7GVQs?|jLfYm|Y2T|RAfWL6BMl~2cYE#$*DX`}w'
            'jC<PDOO)&d)>=Xt-GQckZ>^%rj~q=^m0dId^jp3F?i0-7pa-TDBM}gd@N%GHQ^}DmjZ8T56on1LQBN~@9*IM{ySK1FldJ9Z&d(2O'
            '0^keVQg+Ip(CwM!w(rBJsSB8!bZLm%a}Fc!u#~F3sqG%WeXM_)hVdbz`LGS!7DfSPFzTYn6fj=ncuCU^cb1BZwkr(K_LPAf6nmU;'
            '1y3;iqE*uREtElbd1abJX@%L3=x7>gfMa%|bShz9b~Y@LCsmFqI#uy8YcOx@I?aYx=73}CPBV9d9J{;XSnqt}o=xe%E;P1R_}E_0'
            '$E~Ze!<*)DBx>S(HxenhJU*+ogmrUL%OiGV6U~G!mD?pM>NjyBPVV?3qZ5&As&v!p+8wi+r-vWg3OPD2<d$O5J;1`$MiFBhr!Go%'
            'b4<>usb;lVDze+DJ+;(lSLf+P!fGu#C^4_~b*%&z^{F+e2L^1cDQ>SEDY4rd&M=B|r&p8v0!O^}_HFtuK<icoCk-alri<>#Vo@nx'
            '6>jD`+tShER3G)))wvzS+-CH(ONnl8KCygUhn|*4_UV)Gx=#}H{cP54$}M!Sf4E;vVkCn|t&f5Os~nSxS^P=2Vn`0|`6p=xJ?rOy'
            'grqg)@4J`C9XB7+7Hlflp{7hOX|)*XvVXi|>PWX;D`I@RQ*ewQufH3sn2>jM*!^igX`d(NA?MmLeMtY9SwBK<QTt*u4361{498Gc'
            'X8P3kKsv_`V9GuL0;~K*NsgJ5s7ECjoSC>@qp)hW+lbPI4o{{lQ(XWOb((Qp4%R4-rS6n)_mUg~1d=LCDszMF@n+dLdAwSm@125{'
            '5Bj1!-lm?Z^0=6WX3?u&Iy=TBQM}*b^jWyDz<~`X!AhWc8L@-{y?S_z2kY1oYX_BmW+#`7c`WCR)$F|{))%fRs|B^ue8<)K(Y1fA'
            '%LQbhrYh<9a2)2tz1X3dx~EQb=Fw<(LNpm$`z)|~68IOSw>$_uSYcx--m+E*dIjJ4Mj%eKn#2ad3Evf%@^>v(6M68)>SeN%1><Fv'
            '!n&9!W%Dt|hSnTj7R7sQ#P#lK_F28QQG70}ao1}#_Bi9l9rv2Eu(+;_a>LU?rGC9vw-{z7wHu2Oo)^ojDm)ns{-Z)Qc$AUzJELd5'
            'aTHTt(Kp;$PI%H)MH#Dd=i-h!|5JHy-=c#UYE7bhZTa`YeP@<*ebCW<Jl+z#su-e<?M&_0hjHgF^Y0IofBZf1RoB}2(KRoYDk?m?'
            '3=3W{t@Pk)G3loCB6IIyqm2sMOsQS~f9ifSu=4sycGxhxlF1L(^Ia#3-f;BGc;s_2%!*}{O{h!}$MQ~Y&isyiM@}5{=Mz-1PMQ+c'
            '<fT#k&yXTm?&w?295PH-3BM9AU;c<%42QX>BNC1k*fk5-KGr4*zAa>U?%i38R#vQd`@<TV5iQ>SFk$fCf&ln0pO>+7zF@L}a;`f3'
            'Lx-eRC}5wv+T}=e{yLBPg6uWFNF;m=2Z(eZ_aftCX8`Vd>_*;OTPn1%ANam@g19@+++zhDQQBY}TTfi6_k8EEEvJr--~857yPzwH'
            'A3JiJz{@+^ZRnnv@N_WZo91b&r%hPj?+zd?9lAPn{<1^GD_<9~y0pI8J^0qv(^0kMxMYO_nqe|5SDl%ID8uUmv5roaB%7(-PvAc?'
            'F|FV8ZZ4BHd%7BxCpxtA5rv`BsX*sd&=Udb<o{?{YnL;}!bxAv=vr(}SI;Yqdd0_QPw+9tn_ACLWrfFKvMp#boR_xTL$4EjKmrF|'
            'V^<k0G!VsMoRafXwwwXjd!ultU32XdOLlq{VZJ=0U+L{rlkukDuQ|7>zhGyzjk4@OCCxvqMCYeg=j>W9c$@J-!gP!myhxM?@<7?l'
            'evnZd?P{2RqqvZ|B10YHkjJoVTl@|7vkBcmJI46C>nxebXwFHyqjQ-!9dN2ED2{Tj=Ex-wX&Pbw&X;Sjj4|sYI9KINRINy0-|ME7'
            'cb1RM`@Kaf4Jf9)ysWz#tNIJ-dSfcUFcvbw!*KbGb(UmhIwQCk)L2$&ZwnSaU`eS6N;Z->Rx)SxWi|`<MN>Ca$b?<9m`6&9htqVo'
            '&leQ)UD<W<(d>Ei^Q-We7tb$VhfiNUefjsdFW!DDW~a{DM8jGtv?egV;`=r!CtU9mr%dAn#93fC<*1WSGq!7M1Tx8}!ireN+_iIU'
            '&L39Q++jdF1&peQc9@zqC1PL@a&}#mX}xwXT>e75SDgEvb|_E<cZeva+Q>+Gj_#Hjw83MUY%Om((|t*BRP9Ms0K=(VB<tu#hNt4a'
            'Yn4<Szg3J@Vf9ol5Q~*u@hV)bFj@@SGHO4+pn|Sog7N(X9D|Seg7nL|Fg_Pvg^iHPz7Dzc8<qpW9OdcnQDX9stTAaQmYvF2QrW@f'
            '(N;Y=RFu$cKn0HciO>`wE{{(CgR{{$m{TJ!Wd4M#=g%mG)V)~!L06AgQ?d)Ec`E|h1E<b#$ZwI~HehLqb@Y)UZarVi9g5P$kH}c&'
            '055}b|4g-|<Et#`CjM$mM^smGJnsjpESaa+Ux=r<OwutjcT&FkdKJ}C2DSzk?nsvtT9%JtiA`iKk1T0Q>vSC{`C(oWKT&ykAVx?k'
            'PEc!Q(^zg4I(cVydV*0f={1_L5|*m*OTrPFfVvB^h6C0aYguWSN}Grmbk_)95TXnGN>JK#ANew!=5o)bD_z>Mhl0N|ymd-Q!gv!F'
            'X-UG_Y$)l0(p$`YVv5|LaD}ZY%8KhIL1mo^{$h;v^iWMZ@>N*HT8>ZD+Ld@Zb@KDypT2n^4fYMqDF&{`MIJX|v?=liv_2qOwM%P3'
            'hb35*1U7`fwUsNXO;KiC!1lyh)F)LGC+O_KqE0}W-*63n3DSL?W8P%7#mW)+p2~2idl(ZJwOmG<Z)HISXuXld?LucN{?8MvV+7>e'
            'BmKlSsdElg-y@*uiFdX0)=R=cA(PgK1-sju<0WEG!7Nki9jusj6#U<#l9p9K#VR3W{oo8WnxZ3WO3G&Ua!!eOaWm|pxh^B})?^4H'
            '#*|p8Ophe}C)Wh~+*)9VYk+ljr(k@1X|F+BeN{_1t?vO_COfGcgCYUsTY@1|JwBaPTI8vl0TF}*MrlgG9?zu8rBtTm1J=f`jBImG'
            'SRA6RhAGFGAG9uLBhgw}T37mND=N%HhK=$lfG*Hbs&y-uNEs2YF~{f@{lt+tn<<&E?G?_|rs`o&NIY)w1|$lp;LL5xv|<ep)ifBj'
            '#A5mxg|M|94tf)xD3G@YtimVq;v00f^%`uP4BX*?@6993*5;JF!Np6utseK3urVC<-obV5k8f>OjF#(mG%;|(>wZRFuoH25=nf&P'
            '6Qr);uI&ykFv|b_fGYqQl&@5X%XEdeDzMQMVS%sRG@^@Mg%oakFk<L71=_vP*%MjigeCT31(4R(%{IV{E6{{#4OH^>f=VhJ?Bg~7'
            'IX6jb=0?^4<*cEU^7i&IeVGFBb<ceBRj*Ifx;F@K+>Yy#(rnSbG+Okst!#a27RRAGN&hSbid|jlK%v0TFA&)4CgxR5|8Kft<|gMD'
            'HGO>EeONys7Tg9q11~`<t?|$OWHGf>b@#FFtkYAnYj{LCF5b|A8$c+hRRo*XK0>_)v~tWgqMVXo_^(=t>R+)6)xTiDsfStgk0?H6'
            'yzVPO^`JpKKX3-~FHnUF@4G0klX^n>#zbwkSeWMU*4xKVq(BW<>4zkim&i=CSU{-3CzZEi8Qk5@9;$O?wl-wGr<rx2X*M$izo*$B'
            'Q5EYCu0qmV>-e)Pk{ng%_+P9#iIolFiS~8sPV{GSMDz~5ovPq~td+`uREoxkQA}m80V^tR(J4%~TSWQZbXTxjVkC#ydMyHWQ2=Mg'
            'm8(dwOCD8WG6TwF1qK7%nN=3cgxx3+&>+L4CQ{R~{!9|IUN^~}<Beoogq6l&o;KRSh1g6+s1g+q=1Cx?73@xX^oxU;L0J*!;^YpX'
            '+@IKC!f3<mEF`P{p5zz358)J1@TXaM7Z>QPH`?#S=-`B$E~HUV7^)S#W9#H94n%Fe6thHh<-pRhI6iud5(dXQN6<RfG>Tx;=|ud&'
            'f0>~gY_|k$C+EdR`P^4|-O7aF&;8|N-Dx(6d3guO-d}YSVyHl=l@?<MQvN?DTJHA'
        ),
    },
    'approval': {
        'sha256': '8e6a6e6d92d0b51fe6ce55d06c96ec76680530d63e2335bf2d6961293a8f8357',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;NTXWmSl78o}m;*lutfF{ocPpt=bm>$n5}k0PNM1=!)~j-nATT5mf&dp6%d|@WeZTIW0R|u`Ih)=1!IA(>qo*%l-v*xNT?A>C'
            '#$k}CZB~@0OP%UGDB~<ut7;RZr@?lcXP<F<8I*e1>;3lAM|J<x<WBu^bNS(Ftj7O2xx2sXogVzxJGZMhRZ*(BR)M-3zdye{zaOij'
            ')LT`qf>N#U5}g+6{N4Td7MHZD3Z3I}p@Oiif+X3gLhH0gw+x3mQbAtE3p_AX?}MU<)1{gfVIFVGsW7dhv-`Z#GZm+0#$*>&8uH%g'
            'wLMFz;Cr%EZFG>z1xsGV1$YfMTfEY<lsyoYWx+-(h%f2!R>E8}Z*i$kPqE=emV;8d6KSe2tC9#C`>Yinq{oKpp4Q<VX;$J%Wd+_N'
            'fe(lf?gc8Z=5xq|8G#z^WJ8X+bXw*+C=r`2h&d~Z=8@rcHyfx~Sr+sg-WBy`xlSXUPpdLc<Z7goc+O&h(1S44kjeE;%@w4wtTuQT'
            'Eyuo7m8E!4f7W?VZ@?@levb9yG}r%K#kr1V%Fl8&TLf`JfBap|;<Tt1i#UuyWf}!#fUCV(1QJQMCDPMv5`=n{CE#q<hj#O9lZo0g'
            'HM1I+)n=o=gh^GzpNZp@&h?;2yfI&oD=lgw4Uzb&?KX*x#_xVQKmFJ5zE|0T|I}Yd*<qHJv^{J^cA_JQ`-jQ(<>dNDb^iYS?ai;}'
            'SL*&oUEEyXkFW2Kn7-Km7;@P)!pn7-<x$aWVrpJDEmi?ofqICOC|+u~i{Z<%(VOqcR()frrT$VvoIw@EnOZ=)_~UaNY54<Un}sVK'
            'uBZ0pet!m1r;sTmRfx5k7cC&*mdT$KDy;IH<X{*as3OC1YfSL9z<A7{2da6MrgZWC{%)Y4Z@R>mMHh*(a28Z$w!!j{T{=#|RvgtH'
            'xTT$DRne>E1yd=>O(2N^n@UTp3WGe~5m{&m-t`#dq@e4pT+v0+zpA!L7DUBBZL4__7b{&bQ?wOqj=U+=<ztqwfhzguI0UYFp4VeB'
            'Os9*gta3e_s(7=_ax!#yCOc5k>zQkSrWO3e{`q^6rLvrMQ^tS<bI1M-#p3_3S*qo^ZBP;c``CN@+3OkW`r&Fk{TT*#r_Pk`K@_28'
            'cq6|^y@B#X@buf~4BW$za{5Cg?)7`UHvkoqr(LONrQpcr_+oN5xw)=&JrLjl@I`S@Umd}90=mq&1%hIQ1rcEr)nQzaOU7lPlZEm('
            'S)|EGsyxKcO-7DfusLVs0qHWq<*X{VRS9aqL9!{~m=v>!mwmP|(5rorq|7f2)^KCmrkc@C9z6Sc@6&7ua^hx-SOZ)Ih8btW72d05'
            'Cv0Hx7U-6O`wS+>At`|~2$bWQd6I=|;5H}%vc;glask^lI2>t_J*^^;VL~Lo#+F?SANHWoi}BqZuw(lE>il|Y#XtRd{1NJHHbP2U'
            '!hJ+bi(Eroc{T^`h04K3H4h5j17I;r;&3-2&IpH7!eB%kV&0$&89jcBDh_&6ia0w`w}KCrc-W}eVB;1+R8!~}_O8djj&G-qInpna'
            '%keFl<de5V2=ShY<8!ORO|TQhr5JBt&*)1E1x!wE$MDOd1QQ!lOf2?j9wjA8t{~7Na51mq#Gn&4Sfw>#1A(N0z?IIa6FES!_73&P'
            '_K*Y>Gh42*Lc8?h6lPgSo`5$wQNH#Pkb7Lk;Y!W1gLNlEAcM_06A?>HDJftmvUv}%tswQCUtcOpO7JdrDvrMJuy_F=e(<c}c%L3T'
            '5RGDn9q-dy?DLli+mSJI_8<tbQ161I0D|3)|KkIMd^x=xUqe$<gs02NJxq;Zq^I!DD|NWyWN&J$mCS<m9(e<Kq?19}%e5$9xI^xW'
            'y_ASM#nfU{4{&sIJNa>PeSS4P|8P0E`I8I^v`pazTFlyE2=D4edVzE#NGkZ^)CibTDZhcXC=ScMI{hdAG7_?$H#Un3&SrL`vRVZc'
            'X^gRofeK*+9$I{>5iyX6Je^eOMwcfe^*|O-d>W{eMvJ)oG#n0}aZ$3NY3tqt?tOY5GQo4CbDM3sQ-&a5swge=-dtaOq&%=>;1Z(@'
            'Tt9SN&@lJ|qNBlPm<-I)f)$y5q_?z|f-52>yT<|37BMMxYLgeBTf_g%+&on}06RAMQ;Q^6);_=q79fJ`-%xw)pTCJEfSvtQ?0jHw'
            'u5r=+0mk;^J0sQ(`67$Cp&$Wj*cjhXjE~ooTsU6TR*H=lRl<Snl(Q{tncV&xcZW=B#qtK`lsUNLfI}!Mc^@T<Mu^r(`6nC2h>hX_'
            'i24cs^YBSD-EdA$RaZz$UHYDE&g)ZHczW*3#b$xM!F1w<LrR`BZrHcA3aN2a@058Oaz-BcFZyv`IJL@}O4L15ZDwmEbh&C2O3$|l'
            '$SMaj-E}tbPbo{;7mVZ|9l^BLP<oXj@g$c4sD#1R*qycXX^S<#7RzhUS3d}Wceq4$kmd0*PO(MK@~1^LFXJ-Ng;y`r@CkJs`uREG'
            'ln{*L5<ues*CUsc<}%BImbKK*k(Y@{?qe*nW%ezDU<#u5J>NW{%Dv}|#K9iFw4mZ3NJH)KJvm_4=r{blQ5T%WOQxA`j4KPK3eqAP'
            'P;OFEL}k`3(gbdn;>c*9!Knc&?^~tv0twkGX}4MLYH)95`!yt_m(}x<igJ!;>ZxAf@sr-jNT#{&Xh*`T-(2ZC&+RL7w7&)C`2z;('
            '3E=q$Zn<x2yX~m$RPx-j$r7aIo>C$0(|;pG)tvJJ{H)W0cU6)oAITPK+>rdgUpEd``ioPm2eb^jL-G9K{@v+66xZuR3))7grg9w0'
            '%YA+^;g09^hdK?xr0-Sb;`AS0zdytagudTz^Bmf)-IPjCKb!x)$G6knpf1h)cg=PE#rgHk_2lCGO8xsCsz81vg&n^@9g1MH*1Ljh'
            'gkp<Sy6Bq>qG=NrnG{P1d8nRa!24HQ<Z~~qt_*|ZG1wJqN9k%&-)Cw@Xt`dZdIz2r#3kOWDB$HJ4x}{@9<e}?H={WD7#Gw(<We@*'
            'RN?DL5QPixxLBLAj|@71M5X+rBtDH=1sdcOC&?*aaB4L-#EL2+U=s;=7*(5X(V#k{Ilv<Y^$CI^jN>ztoRAQwBtq^y;%~hT5Hzy9'
            'IP<*$MUjz*Iv1J@kT;w`Ut*Gbx_`lB3~Mh~2Q`3v>M;BnV<sZ7m$ng`2hO@Y3BQjq(##BK!Y`+e#2i5t018Dw&21zLS5>+uaDt+9'
            'KM6MTC>WV1hg7Qi-+%YrzkLsB&EXqu@DetL)t2<>Go_Zq%>8`<*>)S3D&WFjBmZT;AE*QMfh#(j<R9o9+J1%UmT*w{vW36FfXd9Y'
            'PH59it6WjxYx8Snz-+G5>fx9K@^#BbGNqSWEd$$4F*Hr7T>mpP-6o>6aLn^f&D?t4>AG{n@!jNVEVg0Rnfjyqo&}{Zrkr=yNZAy`'
            ')NuF}WtSf3IeWcn*EZz`lCGG0EJlpMJZS{J)1~tDa4EhUkN^?PPdSA)I|PKRgQX@jLfTdVLP{>oC7a<E4;B3q;oBI1AWGV?LA2YQ'
            'YfLhyjX3-87v4NZ(Anr6_Jh5^-PaIyl~<?%iRRz(XXVfJI)PYIT<g+)?%OokGC}EJw-wlMP=jtKjee>9)OWL)PD*9z4U8{P=7A^d'
            '<+eBOZGSJ(=yE<JP*rhA#iu&4R8PpjmzCOO>$fO_f*qa=!eFiDG0=RUXj~*30cQ<_&0_^D2C8o~GN@tu;k}L87fPbZi9-L+ZppBb'
            'Mu9pZiYLJS(C|aML8yect*L(pm*ATST6!5LQJf!T*x=dWsBgD%z*0>2rP-8e>#)CfOD*S~w<jy-<{uW3^k|T(36Yl8U521=<R6IV'
            'p*AiIw_QV>{kMno-Zc#XRvxdv$pG5&<jl`u<bWWX279x6DaI(o?N_b!t$j&4%Zxiw*WYh0_lI#IvHLY^TtKuie&oM7Wn5XYgH@?{'
            'l~l;$gD3`~G-QD_-Z2UA+Z+x|6M^8`Ztz>02#oU6W?8#bbI7?HZ!RC)_<>EE7;|vH4Yg*!2lnc`)T9K`(w}Qe2c{BZUdaktT|=7|'
            ')I$ayU8h<{JcX5nwoU>DxE3Z=WHV-Ni7_djb>PxY+EhARWxxuAdnY=nYpzH)Rawu>YUR{HD_vyNq;D!B+6)RI`r*_`cHxvdi4|2>'
            '9Z-<V{dTeE%qp?s8nD(X+H8QvJxPe0o1D9l8+BLl4MNdYsrT^7)@qzG<vXW8fgL_{nV}xo!Q`&dV827bVV73dT}l0ghu%(F67Y<<'
            '%YoeyErzN1cx2Db=*gErMMfAc!E=w9=Xhj4$0sqPm$$?6e}UsRTmw*oWfJTF&g$f;#n6dm=%oLAqPAHA9jyZ-gs9$Ph$ac@SiK3p'
            'cJSKskIeP^4n+6&1Y{<eZUg7O9wJkVn+rJV<sz&3IKJe9!lc=L+cKzAjw-a!GQtiVlBPJ*%A2T19$V0X49_%Pd8mdrtjQQY$AP(Q'
            'M+)*b$35NvkNDUC`!Atf(`u%)*|w25fkk<n=w`@R560rcq1Ozkn#L1NGN8Q|2OO=)Lw6ob|K?MtN%XqibZn>Nru01bWLhujA6tFU'
            '^4Ln=QHxaFT%Uhb<Le*CS636N`q?B?sGCojK<n$QK>dm=YjhwjSa?q-19dX|J&RLe$P7<D^`D38Rze4nfzhU%I#j5gkgf$WrRJMM'
            'nFbQJ6q&_mt#%diyZAI%qi(3#P$f)$10Ex4(}EOS8hWMN|8PpzLIsXKM#EF9yUvtqpqx--4}x?KjmX_2BZvB+$$V$1z%q5gztjBL'
            '+mC2++(y$W6~oJnf{+A+vlC14gbz5u<ar(yOZ_-I!sAStx!^0uOl;-eo~s3b!QYrk<<oi|*V=xxM8l{*JWx+k7b@Ku(J!hZNCX5-'
            '@3A?U;||aj5=c35z(YS(R3Rn$iz=Zs-mpe+#38#r4+VBm`$yF5*-6JqHv0&`({e6r<Q_$0n4ptjOfShY8XlP(!a$kiTFRFh-!|u?'
            'UCjZZKB*;}mpn@^hw7Ykln6a=zN|FOTk2C3x&EEAakGPWc!Z4W>_jl+gef1M)HAO^B!16lcbpj`Q}YY`(i{TX`S$UC&*@vR298Yl'
            'on8fR4+^F;4^^4R%T*~8yFjw5_Wb`3P5%GNo<KWqc769aUC((;ea-SJO+ZnAZXIvHBm_hNg$i{SQQ!_YI*EtvcOAo)-nW%|-jVf|'
            'p`p4-Lvq|ubIrc0LPW+mjmaC}ZZVYIO&yk5Q8J#Ub-xYuG94+S)IUd9P&*_Xj6wq`0<lRv5b8Bp*e91q)v`*flm!vD%WhNoyipfs'
            'jQhj*-3=U}4$(YllQe^Ds2E5iTw7q_c}vkr>fgw~#GR)xO4TRT5)A+-99TGUT7wmgvZAbH_<Dl$f0-A2LuJpQ8lHYRzMNcLPDZx<'
            'CuT<_J|~FlB(9@n9nPbebap6!y8+)WgLAZkIb!*NKyAXi1a@w`#La0hM0r;G;b?xe!EhkH+eB9){Gp|H(#43sBD~Nd@j6{@^jeXf'
            '@3Oq$`Ne)N>G#!|ww&&z4l?_eNXgaBHxxT$e7OWfadEcdCfTzlb;NOpG|Srh{b7<l>Kv^7?X1Otz5cBLvDh~m0Rxt;nsXbBecg0('
            '=p`qy@}G_{_%QmH?_mtXVZWn>g=fJPMtJw({QBpK1&191vH#?5!=XYAm108*8mfu#ARI>VAvh_rGzS3b6rn7&y(gGcmWXw{Jp^{J'
            'z#N`nF^^3V6zpdA)ZhPquO0ue*8WVL*b$*yHXf@JNuD`;dJU0vipHLs-@+q!{M^Pmd-uV%GpPP(V*J5p=$K?dCrk0@!jpkc{gz-s'
            '=O#^hhIsqr1mEW_JzhI)j`X$LGBp<?Zaob4?|>$?WtzVZJrmGa-kvfZrrD!!4;l}vGNjI026RC7?YpfUE&{NSe^QdwY04ZqpN<u-'
            '_qZ(@Q`*!yoSC<^kDwndZVvU;>{0c-lA%>9$ae_Y2BkbfU8&5c9&*>PW)4yXWs~u<xVp#p>e>c{Bu|Iw7qN$vXX`Ze$0b}UxA7ml'
            'FsDYW#rV#YshL6dyR>9n%(wY>p;dEh`W+CLNfz2=(@WG0m!M_^XXfInAcU&l@s<dG+rhQwe14(s?$58rtOU33!Kc~C3XWGAb&1gw'
            'Va<^sI-_`gb$3Ij3~WvlQOYGQ)|{|b%Z$$~%PgPghcpj#XPhp#Nnre!tF6;hY50BSsbf|xSI)Nz^0fv2-V{0^GZe)d;kBy*IR?mQ'
            'qaC+4X9DDio%6Q=m=TuOhwTnGUn%N36rJuOz3;(mK~-H-Fs{ZjTtZ<cb$bV_<?2ZvMMvN3b}fZ`ujjt1=AzXrtzBzxPS63r_PpR2'
            'ymU$uq5a~vJ7vgzlLFC29eJJ90Wxl!SJGnYgeXZIw3?gD6cK(v6%-19_$y-J4naSlVybs^*5nGa)yK9z{Edz4*W1zut)i%cbRBoH'
            's+BH+t%P14o`B5ri&j8q0)<^SA+#DgkPN=2DU(_JIKDtSGWoBOWssB*!M`QFA`>n)drA7j^KI^so;_JE>Rb*u!J9s_sa})i4b?}J'
            'v6<`>Wv3zaNA&nc{pw&um0Cmvr`I?4s`<Xnw2%b%Ms3{V)`}s=*jYEVn{)%P+8j1^&h|Rb)2Z6Ox<GtQyG@PQC|FQcG^!<2E}-GS'
            'EghZKD)F>_sY;|wz1t-|UNjyX2)}}Q0^J?R+uJ;6vdZ-|0~s-8_zV|G+&Tql-^EGN!D6tRD!thJZSLAO?+bza)dujb`AV^Lz5r9d'
            'o!?*lG`^J6OTy|v{WiJ(>E^?|^JPR_)(uiV1%*1Nu$>ZaY8av>g_ch^#d8SrlMHrUp8~S3BK5ERW=PE58IGt4*5q_vkh-;Ys55KH'
            'b7l@t(MPXxV8;2(mA~R5$K`&Dnn@IyQOKIL;~0Mj5Y*Emd4+Ts`I{n1IsV=>CMl6|KV0Hr)ZRC;NBiAxZ4K|he%$^nRQkL1hiKeq'
            'uTwd3Is=0VeWJw&CHH!otrp<R?MCOZrrl_3-MZ(v3%2Fap5tq+42xdf5cqPhQq?oSzo45jxxN@*jei-dpU*#DPFfYR)63uu`cE_+'
            'KEhf5g+7J>6w%?)z6A{uo!-S>g<RXo!3x>leehcD(k)i)WBbLbV-DWPwV^}L{O^j@BceN4jJ{UgKAV6mE>3-athYmrPQ$gvYV-ZI'
            'pfw#lo&rXuT-%zHuI)m$thR}M5ZtE!Kh@BBk)<d$kQsz}#FWU{`5<=P^=(cwQ}h=eWGd`8pX_$(YBGerOtzxzF*OOs7Glvn+LGA_'
            'm&WpqgN#^cTI77$Ur2^gA0^p8&g%{81LJ4*r1GKjz4d0y@oCzrFAuc_p+`5=+#t#Jd;bGEZ&}L'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
render_utils = _local_pipeline_modules['render_utils']
video_analysis = _local_pipeline_modules['video_analysis']
approval = _local_pipeline_modules['approval']



In [ ]:
ffmpeg_bin, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
print('FFmpeg:', ffmpeg_bin)
print('System ffprobe:', ffprobe_bin or 'bulunamadi (probe_media() ffmpeg fallback kullanacak)')


def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None


## 1. Girdi dosyalarını yükle: `hot_moments.json`, `selected_script.json`, `selected_voiceover.json`

In [ ]:
def load_json_artifact(filename, override_path=''):
    candidates = [
        override_path,
        str(RESULTS_DIR / filename),
        f'results/{filename}',
        f'../results/{filename}',
        f'/content/results/{filename}',
    ]
    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f'{filename} bulunamadi. Aranan yollar: ' + ', '.join(c for c in candidates if c)
        )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f), path

hot_moments_data, hot_moments_path = load_json_artifact('hot_moments.json')
selected_script, selected_script_path = load_json_artifact('selected_script.json')
selected_voiceover, selected_voiceover_path = load_json_artifact('selected_voiceover.json')

# Kanonik sema dogrulamasi (Notebook 01/02 ve master notebook ile ayni fonksiyon).
schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
if schema_errors:
    raise ValueError(f'hot_moments.json semasi gecersiz ({hot_moments_path}):\n- ' + '\n- '.join(schema_errors))

print('hot_moments.json      :', hot_moments_path, '| status:', hot_moments_data.get('status'))
print('selected_script.json  :', selected_script_path, '| status:', selected_script.get('status'))
print('selected_voiceover.json:', selected_voiceover_path, '| status:', selected_voiceover.get('status'))

if selected_script.get('status') != 'approved':
    print(f"UYARI: selected_script.json status='{selected_script.get('status')}' (approved degil)")
if selected_voiceover.get('status') != 'approved':
    print(f"UYARI: selected_voiceover.json status='{selected_voiceover.get('status')}' (approved degil)")
print('Not: onaylanmamis girdilerle render yine calisir (inceleme icin) ama nihai durum')
print('     review_required/failed kalir, ASLA approved/production_ready olmaz.')


## 2. Render yapılandırması ve onay bayrağı

Tüm render mantığı tek bir kanonik çağrıda (`render_utils.run_render_pipeline`) toplanmıştır: aralık kesme, altyazı hizalama (gerçek seslendirme sesinden `faster-whisper` ile; başarısız olursa orantılı fallback), dikey kadraj + altyazı yakma + ses miksleme (tek `ffmpeg` geçişi), teknik doğrulama ve onay kapısı değerlendirmesi.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' veya 'center_crop'
TARGET_WIDTH = 1080
TARGET_HEIGHT = 1920
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi

DUCKING_ENABLED = True
ORIGINAL_AUDIO_BASE_VOLUME = 0.35
NARRATION_VOLUME = 1.0
DUCK_RATIO = 8

SUBTITLE_FONTSIZE = 20
SUBTITLE_MARGIN_V = 90
SUBTITLE_MAX_CHARS_PER_CUE = 42

SOURCE_VIDEO_PATH_OVERRIDE = ''  # zorla belirli bir ham kaynak video vermek istersen buraya yaz
print('Render modu:', RENDER_MODE, '| onizleme mi:', PREVIEW_MODE)
print("NOT: onay bu hucrede DEGIL, bolum 5'te (videoyu izledikten sonra) verilir.")


## 3. Render çalıştır (kanonik `render_utils.run_render_pipeline`)

In [ ]:
render_manifest = render_utils.run_render_pipeline(
    hot_moments_data, selected_script, selected_voiceover,
    hot_moments_path, selected_script_path, selected_voiceover_path,
    render_dir=RENDER_DIR,
    render_mode=RENDER_MODE, target_width=TARGET_WIDTH, target_height=TARGET_HEIGHT,
    preview_mode=PREVIEW_MODE, ducking_enabled=DUCKING_ENABLED,
    original_audio_base_volume=ORIGINAL_AUDIO_BASE_VOLUME, narration_volume=NARRATION_VOLUME,
    duck_ratio=DUCK_RATIO, subtitle_fontsize=SUBTITLE_FONTSIZE,
    subtitle_margin_v=SUBTITLE_MARGIN_V, subtitle_max_chars_per_cue=SUBTITLE_MAX_CHARS_PER_CUE,
    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
)

print('Kaynak cozumlemesi:', render_manifest['inputs']['source_description'])
print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
      f"({render_manifest['subtitles']['cue_count']} satir)")
print('Render dosyasi:', render_manifest['output']['final_video_path'],
      f"({render_manifest['output']['render_latency_sec']}s)")
print()
print('Teknik dogrulama (passed):', render_manifest['output']['validation']['passed'],
      '| probe backend:', render_manifest['output']['validation']['details']['probe_backend'])
print(json.dumps(render_manifest['output']['validation']['checks'], indent=2))
print()
print('Onay oncesi kosullar:', json.dumps(render_manifest['approval']['conditions'], indent=2))
if render_manifest['approval']['failed_conditions']:
    print('Saglanmayan kosul(lar):', render_manifest['approval']['failed_conditions'])
print('Render durumu:', render_manifest['status'], '-', render_manifest['status_reason'])


## 5. Render'ı izle ve ONAYLA (ayrı adım)

Videoyu **izledikten sonra** aşağıda yazdırılan SHA-256'yı `RENDER_REVIEWED_SHA256` alanına yapıştır.

Render onayı yalnızca **tüm** koşullar sağlandığında verilebilir: teknik doğrulama geçmiş **ve**
script onaylı **ve** seslendirme onaylı **ve** seslendirme-script hash eşleşmesi sağlanmış olmalı.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır.

Sadece izlediğini kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap — bu `status`'ü değiştirmez.

In [ ]:
video_path = render_manifest['output']['final_video_path']
try:
    from IPython.display import Video, display as _display
    _display(Video(video_path, embed=True, width=320))
except Exception as exc:
    print('Video oynatici gosterilemedi:', exc)
    print('Dosyayi elle izle:', video_path)

render_review_info = approval.describe_for_review('render', render_manifest)
print('Icerik SHA-256:', render_review_info['content_sha256'])
if not render_review_info['reviewable']:
    print()
    print('!!! FINAL ONAY ENGELLI !!!')
    print(render_review_info['blocker'])
    for condition in render_review_info.get('blocking_conditions', []):
        print('  -', condition)

RENDER_REVIEWED_SHA256 = ''   # yukaridaki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''
RECORD_RENDER_REVIEW_ONLY = False  # True: sadece "izledim" kaydi, status DEGISMEZ

if not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        print('Render ONAYLANDI (status:', render_manifest['status'], ')')
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## 6. Dışa aktarım: `render_manifest.json`

In [ ]:
render_manifest_path = RENDER_DIR / 'render_manifest.json'
with open(render_manifest_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_manifest_path, '(status:', render_manifest['status'], ')')

render_validation_path = RENDER_DIR / 'render_validation.json'
with open(render_validation_path, 'w', encoding='utf-8') as f:
    json.dump(render_manifest['output']['validation'], f, ensure_ascii=False, indent=2)
print('Kaydedildi:', render_validation_path)

print('HATIRLATMA: bu notebook hicbir yere yukleme yapmadi (not_uploaded='
      f"{render_manifest['not_uploaded']}).")
print('Yayin, ayri ve elle onayli bir adimdir.')
